In [ ]:
import os, json, math, zipfile, copy, itertools, sys
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any
import numpy as np

import csv
import importlib.util
import tempfile
from collections import Counter

try:
    import onnx, onnx.helper as oh, onnx.numpy_helper as onh
    from onnx import TensorProto
except ImportError:
    os.system("pip install onnx -q")
    import onnx, onnx.helper as oh, onnx.numpy_helper as onh
    from onnx import TensorProto
try:
    import onnxruntime as ort
except ImportError:
    os.system("pip install onnxruntime onnx-tool -q")
    import onnxruntime as ort
    
import numpy as np
import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
import onnxruntime as ort


def resolve_task_dir() -> Path:
    candidates = [
        Path('/kaggle/input/competitions/neurogolf-2026'),
        Path('.'),
    ]
    for candidate in candidates:
        if candidate.exists() and list(candidate.glob('task*.json')):
            return candidate
    for root in candidates:
        if not root.exists():
            continue
        for path in root.rglob('task001.json'):
            return path.parent
    raise FileNotFoundError('Could not find task*.json files')


TASK_DIR = resolve_task_dir()
WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

In [ ]:
%%writefile starter_solver.py

import os, json, zipfile, math, copy, sys, time, traceback, random, tempfile, importlib.util
import numpy as np
from pathlib import Path
from itertools import product as iprod

import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto

try:
    import onnxruntime as ort
except ImportError:
    os.system("pip install onnxruntime -q")
    import onnxruntime as ort

try:
    import onnx_tool
except ImportError:
    os.system("pip install onnx-tool -q")

# ============================================================
# CONFIG
# ============================================================
TASK_DIR = Path("/kaggle/input/competitions/neurogolf-2026")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
C, H, W = 10, 30, 30
HW = H * W
CHW = C * H * W
MAX_ARC_GEN_VALIDATE = 9999  # FIX #2: validate ALL arc-gen, not just 50

# ============================================================
# EXACT SCORING via neurogolf_utils
# ============================================================
_UTILS = None
def _load_utils():
    global _UTILS
    if _UTILS is not None: return _UTILS
    for d in [TASK_DIR, Path('/kaggle/input/competitions/neurogolf-2026')]:
        up = d / 'neurogolf_utils' / 'neurogolf_utils.py'
        if up.exists():
            spec = importlib.util.spec_from_file_location('ng', up)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            _UTILS = mod
            return mod
    return None

def exact_cost_via_utils(model):
    """Use neurogolf_utils.score_network for EXACT cost."""
    utils = _load_utils()
    if utils is None: return None
    try:
        with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as f:
            f.write(model.SerializeToString()); f.flush(); fname = f.name
        macs, mem, params = utils.score_network(fname)
        os.unlink(fname)
        return int(macs + mem + params)
    except:
        try: os.unlink(fname)
        except: pass
        return None

# ============================================================
# DATA UTILS
# ============================================================
def load_task(path):
    with open(path) as f:
        return json.load(f)

def grid_to_tensor(grid):
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    if h > H or w > W:
        return t
    for v in range(C):
        t[0, v, :h, :w] = (g == v).astype(np.float32)
    return t

def tensor_to_grid(t, out_h, out_w):
    arr = t[0]
    out_h = min(out_h, H)
    out_w = min(out_w, W)
    slice_ = arr[:, :out_h, :out_w]
    max_vals = slice_.max(axis=0)
    argmax = slice_.argmax(axis=0)
    grid = np.where(max_vals < 0.5, 0, argmax).astype(int)
    return grid.tolist()

def shapes_match(p):
    return np.array(p["input"]).shape == np.array(p["output"]).shape

# ============================================================
# MODEL CHECK + COST
# ============================================================
def check_model_correct(model, pairs):
    try:
        buf = model.SerializeToString()
        so = ort.SessionOptions(); so.log_severity_level = 3
        sess = ort.InferenceSession(buf, sess_options=so, providers=["CPUExecutionProvider"])
        for p in pairs:
            inp = grid_to_tensor(p["input"])
            out_g = np.array(p["output"])
            oh_, ow_ = out_g.shape
            pred = sess.run(None, {"input": inp})[0]
            pred_grid = tensor_to_grid(pred, oh_, ow_)
            if pred_grid != p["output"]:
                return False
        return True
    except Exception:
        return False

def estimate_model_cost(model):
    """FIX #1: Don't count elementwise ops as MACs — organizers only count Conv/Gemm/MatMul."""
    try:
        # First try exact scoring
        ec = exact_cost_via_utils(model)
        if ec is not None:
            return ec

        # Fallback estimate
        total_params = 0
        total_bytes = 0
        tensors = {}

        for init in model.graph.initializer:
            arr = onh.to_array(init)
            tensors[init.name] = arr
            total_params += arr.size
            total_bytes += arr.nbytes
        for node in model.graph.node:
            if node.op_type == "Constant":
                for attr in node.attribute:
                    if attr.t and (attr.t.raw_data or list(attr.t.float_data)
                                   or list(attr.t.int64_data) or list(attr.t.int32_data)):
                        arr = onh.to_array(attr.t)
                        if node.output:
                            tensors[node.output[0]] = arr
                        total_params += arr.size
                        total_bytes += arr.nbytes

        total_macs = 0
        for node in model.graph.node:
            if node.op_type == "Conv":
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    if w.ndim == 4:
                        c_out, c_in, kh, kw = w.shape
                        total_macs += c_out * c_in * kh * kw * H * W
            elif node.op_type == "Gemm":
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    if w.ndim == 2:
                        total_macs += w.shape[0] * w.shape[1]
            # FIX: Do NOT count Mul/Add/Max/Relu as MACs

        return total_params + total_bytes + total_macs
    except Exception:
        return float('inf')

# ============================================================
# ONNX BUILDERS
# ============================================================

def make_identity_onnx():
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    node = oh.make_node("Identity", ["input"], ["output"])
    graph = oh.make_graph([node], "identity", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

# ============================================================
# CHEAP PRIMITIVES (Transpose/Slice/Pad — 0 MACs)
# ============================================================

def _const_node(name, arr):
    t = onh.from_array(arr, name=name + "_v")
    return oh.make_node("Constant", [], [name], value=t)

def _slice_reverse_nodes(axes, dims, in_name, out_name, suf=""):
    starts = [d - 1 for d in dims]
    ends = [-d - 1 for d in dims]
    steps = [-1] * len(dims)
    return [
        _const_node(f"rs_s{suf}", np.asarray(starts, dtype=np.int64)),
        _const_node(f"rs_e{suf}", np.asarray(ends, dtype=np.int64)),
        _const_node(f"rs_a{suf}", np.asarray(axes, dtype=np.int64)),
        _const_node(f"rs_p{suf}", np.asarray(steps, dtype=np.int64)),
        oh.make_node("Slice",
                     [in_name, f"rs_s{suf}", f"rs_e{suf}", f"rs_a{suf}", f"rs_p{suf}"],
                     [out_name]),
    ]

def _pad_to_canvas_nodes(in_name, out_name, out_h, out_w, suf=""):
    pads = np.asarray([0, 0, 0, 0, 0, 0, H - out_h, W - out_w], dtype=np.int64)
    return [
        _const_node(f"pd{suf}", pads),
        oh.make_node("Pad", [in_name, f"pd{suf}"], [out_name], mode="constant"),
    ]

def _crop_nodes(in_name, out_name, h, w, suf=""):
    return [
        _const_node(f"cs{suf}", np.asarray([0, 0], dtype=np.int64)),
        _const_node(f"ce{suf}", np.asarray([h, w], dtype=np.int64)),
        _const_node(f"ca{suf}", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", [in_name, f"cs{suf}", f"ce{suf}", f"ca{suf}"], [out_name]),
    ]

def _build_model_from_nodes(nodes, name):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    graph = oh.make_graph(nodes, name, [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_cheap_rot180_framed(h, w):
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([2, 3], [h, w], "content", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", h, w, "p")
    return _build_model_from_nodes(nodes, "rot180_framed")

def make_cheap_rot90cw_framed(h, w):
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([3], [h], "t", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "rot90cw_framed")

def make_cheap_rot90ccw_framed(h, w):
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2], [w], "t", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "rot90ccw_framed")

def make_cheap_flip_h_framed(h, w):
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([2], [h], "content", "flipped", "r")
    nodes += _pad_to_canvas_nodes("flipped", "output", h, w, "p")
    return _build_model_from_nodes(nodes, "flip_h_framed")

def make_cheap_flip_w_framed(h, w):
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([3], [w], "content", "flipped", "r")
    nodes += _pad_to_canvas_nodes("flipped", "output", h, w, "p")
    return _build_model_from_nodes(nodes, "flip_w_framed")

def make_cheap_transpose_framed(h, w):
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _pad_to_canvas_nodes("t", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "transpose_framed")

def make_cheap_anti_transpose_framed(h, w):
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2, 3], [w, h], "t", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "anti_transpose_framed")

def make_cheap_crop(r0, c0, out_h, out_w):
    nodes = [
        _const_node("cs", np.asarray([r0, c0], dtype=np.int64)),
        _const_node("ce", np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const_node("ca", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "cs", "ce", "ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas_nodes("cropped", "output", out_h, out_w, "p")
    return _build_model_from_nodes(nodes, "crop_net")

def make_cheap_tile(tr, tc, in_h, in_w):
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [
        _const_node("tile_r", np.asarray([1, 1, tr, tc], dtype=np.int64)),
        oh.make_node("Tile", ["content", "tile_r"], ["tiled"]),
    ]
    nodes += _pad_to_canvas_nodes("tiled", "output", in_h * tr, in_w * tc, "p")
    return _build_model_from_nodes(nodes, "tile_net")

def make_cheap_upscale(f, in_h, in_w):
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [
        _const_node("rsz_roi", np.asarray([], dtype=np.float32)),
        _const_node("rsz_scales", np.asarray([], dtype=np.float32)),
        _const_node("rsz_sizes", np.asarray([1, C, in_h * f, in_w * f], dtype=np.int64)),
        oh.make_node("Resize", ["content", "rsz_roi", "rsz_scales", "rsz_sizes"], ["upscaled"],
                     mode="nearest", coordinate_transformation_mode="asymmetric", nearest_mode="floor"),
    ]
    nodes += _pad_to_canvas_nodes("upscaled", "output", in_h * f, in_w * f, "p")
    return _build_model_from_nodes(nodes, "upscale_net")

def make_cheap_concat_flip_w(in_h, in_w):
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([3], [in_w], "content", "flipped", "fw")
    nodes += [oh.make_node("Concat", ["content", "flipped"], ["concated"], axis=3)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "concat_flip_w")

def make_cheap_concat_flip_h(in_h, in_w):
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([2], [in_h], "content", "flipped", "fh")
    nodes += [oh.make_node("Concat", ["content", "flipped"], ["concated"], axis=2)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h * 2, in_w, "p")
    return _build_model_from_nodes(nodes, "concat_flip_h")

def make_cheap_quadrant_mirror(in_h, in_w):
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([3], [in_w], "content", "fw", "fw")
    nodes += [oh.make_node("Concat", ["content", "fw"], ["top_row"], axis=3)]
    nodes += _slice_reverse_nodes([2], [in_h], "content", "fh", "fh")
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], "content", "fr", "fr")
    nodes += [oh.make_node("Concat", ["fh", "fr"], ["bot_row"], axis=3)]
    nodes += [oh.make_node("Concat", ["top_row", "bot_row"], ["full"], axis=2)]
    nodes += _pad_to_canvas_nodes("full", "output", in_h * 2, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "quadrant_mirror")

def make_cheap_self_concat_h(in_h, in_w):
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [oh.make_node("Concat", ["content", "content"], ["concated"], axis=2)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h * 2, in_w, "p")
    return _build_model_from_nodes(nodes, "self_concat_h")

def make_cheap_self_concat_w(in_h, in_w):
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [oh.make_node("Concat", ["content", "content"], ["concated"], axis=3)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "self_concat_w")

def make_cheap_framed_translation(grid_h, grid_w, dr, dc):
    src_r0 = max(0, -dr); src_r1 = min(grid_h, grid_h - dr)
    src_c0 = max(0, -dc); src_c1 = min(grid_w, grid_w - dc)
    dst_r0 = max(0, dr);  dst_c0 = max(0, dc)
    rh = src_r1 - src_r0; rw = src_c1 - src_c0
    if rh <= 0 or rw <= 0: return None
    nodes = [
        _const_node("fts", np.asarray([src_r0, src_c0], dtype=np.int64)),
        _const_node("fte", np.asarray([src_r1, src_c1], dtype=np.int64)),
        _const_node("fta", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "fts", "fte", "fta"], ["fsliced"]),
    ]
    pads = np.asarray([0, 0, dst_r0, dst_c0, 0, 0, H - dst_r0 - rh, W - dst_c0 - rw], dtype=np.int64)
    nodes += [_const_node("fpp", pads), oh.make_node("Pad", ["fsliced", "fpp"], ["output"], mode="constant")]
    return _build_model_from_nodes(nodes, "framed_translate")

def make_cheap_downscale(f, in_h, in_w):
    out_h = in_h // f; out_w = in_w // f
    nodes = [
        _const_node("ds_s", np.asarray([0, 0], dtype=np.int64)),
        _const_node("ds_e", np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node("ds_a", np.asarray([2, 3], dtype=np.int64)),
        _const_node("ds_p", np.asarray([f, f], dtype=np.int64)),
        oh.make_node("Slice", ["input", "ds_s", "ds_e", "ds_a", "ds_p"], ["downsampled"]),
    ]
    nodes += _pad_to_canvas_nodes("downsampled", "output", out_h, out_w, "p")
    return _build_model_from_nodes(nodes, "downscale")

def make_cheap_row_broadcast(in_h, in_w, row_idx=0):
    nodes = [
        _const_node("rb_s", np.asarray([row_idx], dtype=np.int64)),
        _const_node("rb_e", np.asarray([row_idx + 1], dtype=np.int64)),
        _const_node("rb_a", np.asarray([2], dtype=np.int64)),
        oh.make_node("Slice", ["input", "rb_s", "rb_e", "rb_a"], ["one_row"]),
        _const_node("rb_r", np.asarray([1, 1, in_h, 1], dtype=np.int64)),
        oh.make_node("Tile", ["one_row", "rb_r"], ["tiled"]),
        _const_node("rb_cs", np.asarray([0, 0], dtype=np.int64)),
        _const_node("rb_ce", np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node("rb_ca", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["tiled", "rb_cs", "rb_ce", "rb_ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas_nodes("cropped", "output", in_h, in_w, "p")
    return _build_model_from_nodes(nodes, "row_broadcast")

def make_cheap_col_broadcast(in_h, in_w, col_idx=0):
    nodes = [
        _const_node("cb_s", np.asarray([col_idx], dtype=np.int64)),
        _const_node("cb_e", np.asarray([col_idx + 1], dtype=np.int64)),
        _const_node("cb_a", np.asarray([3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "cb_s", "cb_e", "cb_a"], ["one_col"]),
        _const_node("cb_r", np.asarray([1, 1, 1, in_w], dtype=np.int64)),
        oh.make_node("Tile", ["one_col", "cb_r"], ["tiled"]),
        _const_node("cb_cs", np.asarray([0, 0], dtype=np.int64)),
        _const_node("cb_ce", np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node("cb_ca", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["tiled", "cb_cs", "cb_ce", "cb_ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas_nodes("cropped", "output", in_h, in_w, "p")
    return _build_model_from_nodes(nodes, "col_broadcast")

def make_cheap_border_add(grid_h, grid_w, border, fill_color):
    oh_ = grid_h + 2 * border; ow_ = grid_w + 2 * border
    if oh_ > H or ow_ > W: return None
    nodes = _crop_nodes("input", "content", grid_h, grid_w, "pb0")
    nodes += [_const_node("pb_pads", np.asarray([0, 0, border, border, 0, 0, border, border], dtype=np.int64)),
              oh.make_node("Pad", ["content", "pb_pads"], ["padded"], mode="constant")]
    mask = np.zeros((1, C, oh_, ow_), dtype=np.float32)
    if 0 <= fill_color < C:
        mask[0, fill_color, :border, :] = 1.0; mask[0, fill_color, -border:, :] = 1.0
        mask[0, fill_color, :, :border] = 1.0; mask[0, fill_color, :, -border:] = 1.0
    nodes += [_const_node("pb_mask", mask), oh.make_node("Add", ["padded", "pb_mask"], ["bordered"])]
    nodes += _pad_to_canvas_nodes("bordered", "output", oh_, ow_, "pbf")
    return _build_model_from_nodes(nodes, "border_add")

def make_cheap_concat_rot180_w(in_h, in_w):
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], "content", "rotated", "r")
    nodes += [oh.make_node("Concat", ["content", "rotated"], ["concated"], axis=3)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "concat_rot180_w")

def make_cheap_crop_then_color(r0, c0, out_h, out_w, color_map):
    nodes = [
        _const_node("ctc_s", np.asarray([r0, c0], dtype=np.int64)),
        _const_node("ctc_e", np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const_node("ctc_a", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "ctc_s", "ctc_e", "ctc_a"], ["cropped"]),
    ]
    inv = {d: s for s, d in color_map.items()}
    idx = np.arange(C, dtype=np.int64); mask = np.ones(C, dtype=np.float32)
    src_away = {s for s, d in color_map.items() if s != d}
    for d in range(C):
        if d in inv: idx[d] = inv[d]
        elif d in src_away: idx[d] = 0; mask[d] = 0.0
    nodes += [_const_node("gi_ctc", idx)]
    if mask.min() == 1.0:
        nodes += [oh.make_node("Gather", ["cropped", "gi_ctc"], ["colored"], axis=1)]
    else:
        nodes += [oh.make_node("Gather", ["cropped", "gi_ctc"], ["g_ctc"], axis=1),
                  _const_node("cm_ctc", mask.reshape(1, C, 1, 1)),
                  oh.make_node("Mul", ["g_ctc", "cm_ctc"], ["colored"])]
    nodes += _pad_to_canvas_nodes("colored", "output", out_h, out_w, "pctc")
    return _build_model_from_nodes(nodes, "crop_color")

# ============================================================
# GATHER-BASED BUILDERS
# ============================================================
SAFE_PAD_IDX = HW - 1

def _compact_mask(mask):
    if mask.ndim == 4 and mask.shape[0] == 1 and mask.shape[1] == C:
        if np.all(mask[0, 0:1] == mask[0, :]):
            return mask[:, 0:1].copy()
    return mask

def make_gather_onnx(gather_indices, mask=None):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])
    gi_tensor = onh.from_array(gather_indices.astype(np.int32), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)

    if mask is not None:
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"])
        mask = _compact_mask(mask)
        mask_tensor = onh.from_array(mask.astype(np.float32), name="mask")
        mask_node = oh.make_node("Constant", [], ["mask"], value=mask_tensor)
        mul_node = oh.make_node("Mul", ["gathered_4d", "mask"], ["output"])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
                 out_shape_node, reshape_back, mask_node, mul_node]
    else:
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]

    graph = oh.make_graph(nodes, "gather_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_const_onnx(const_output):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    const_tensor = onh.from_array(const_output.astype(np.float32), name="const_out")
    const_node = oh.make_node("Constant", [], ["const_out"], value=const_tensor)
    id_node = oh.make_node("Identity", ["const_out"], ["output"])
    graph = oh.make_graph([const_node, id_node], "const_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_conv1x1_onnx(weight):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                             kernel_shape=[1, 1], pads=[0, 0, 0, 0])
    graph = oh.make_graph([w_node, conv_node], "conv1x1", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_channel_gather_onnx(gather_ch):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    gi_tensor = onh.from_array(gather_ch.astype(np.int32), name="gi")
    gi_node = oh.make_node("Constant", [], ["gi"], value=gi_tensor)
    gather = oh.make_node("Gather", ["input", "gi"], ["output"], axis=1)
    graph = oh.make_graph([gi_node, gather], "ch_gather", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_conv_onnx(weight, bias=None, kernel_size=3):
    pad = kernel_size // 2
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    if bias is not None:
        b_tensor = onh.from_array(bias.astype(np.float32), name="conv_bias")
        b_node = oh.make_node("Constant", [], ["conv_bias"], value=b_tensor)
        conv_node = oh.make_node("Conv", ["input", "conv_weight", "conv_bias"], ["output"],
                                 kernel_shape=[kernel_size, kernel_size], pads=[pad]*4)
        nodes = [w_node, b_node, conv_node]
    else:
        conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                                 kernel_shape=[kernel_size, kernel_size], pads=[pad]*4)
        nodes = [w_node, conv_node]
    graph = oh.make_graph(nodes, "conv_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_spatial_then_channel_gather_onnx(spatial_idx, channel_idx):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    nodes = [
        oh.make_node("Constant", [], ["shape_chw"], value=shape_chw),
        oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"]),
        oh.make_node("Constant", [], ["si"], value=onh.from_array(spatial_idx.astype(np.int32), name="si")),
        oh.make_node("Gather", ["inp_chw", "si"], ["spatial_out"], axis=2),
        oh.make_node("Constant", [], ["out_shape"], value=onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")),
        oh.make_node("Reshape", ["spatial_out", "out_shape"], ["spatial_4d"]),
        oh.make_node("Constant", [], ["ci"], value=onh.from_array(channel_idx.astype(np.int32), name="ci")),
        oh.make_node("Gather", ["spatial_4d", "ci"], ["output"], axis=1),
    ]
    graph = oh.make_graph(nodes, "spatial_ch_gather", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_gather_then_conv1x1_onnx(gather_indices, conv_weight, mask=None):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["shape_chw"], value=onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")),
        oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"]),
        oh.make_node("Constant", [], ["gather_idx"], value=onh.from_array(gather_indices.astype(np.int32), name="gather_idx")),
        oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2),
        oh.make_node("Constant", [], ["out_shape"], value=onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")),
        oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"]),
    ]
    w_tensor = onh.from_array(conv_weight.astype(np.float32), name="conv_weight")
    if mask is not None:
        mask = _compact_mask(mask)
        nodes += [
            oh.make_node("Constant", [], ["mask"], value=onh.from_array(mask.astype(np.float32), name="mask")),
            oh.make_node("Mul", ["gathered_4d", "mask"], ["masked"]),
            oh.make_node("Constant", [], ["conv_weight"], value=w_tensor),
            oh.make_node("Conv", ["masked", "conv_weight"], ["output"], kernel_shape=[1, 1], pads=[0, 0, 0, 0]),
        ]
    else:
        nodes += [
            oh.make_node("Constant", [], ["conv_weight"], value=w_tensor),
            oh.make_node("Conv", ["gathered_4d", "conv_weight"], ["output"], kernel_shape=[1, 1], pads=[0, 0, 0, 0]),
        ]
    graph = oh.make_graph(nodes, "gather_conv1x1", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_symmetry_completion_onnx(gather_index_list, mask=None):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    ch_mask = np.ones((1, C, 1, 1), dtype=np.float32); ch_mask[0, 0, 0, 0] = 0.0
    nodes = [
        oh.make_node("Constant", [], ["ch_mask"], value=onh.from_array(ch_mask, name="ch_mask")),
        oh.make_node("Mul", ["input", "ch_mask"], ["input_nobg"]),
        oh.make_node("Constant", [], ["shape_chw"], value=onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")),
        oh.make_node("Reshape", ["input_nobg", "shape_chw"], ["inp_chw"]),
        oh.make_node("Constant", [], ["out_shape"], value=onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")),
    ]
    reshape_names = []
    for i, gi in enumerate(gather_index_list):
        gn = f"gi_{i}"; rn = f"r_{i}"
        nodes += [
            oh.make_node("Constant", [], [gn], value=onh.from_array(gi.astype(np.int32), name=gn)),
            oh.make_node("Gather", ["inp_chw", gn], [f"g_{i}"], axis=2),
            oh.make_node("Reshape", [f"g_{i}", "out_shape"], [rn]),
        ]
        reshape_names.append(rn)
    if len(reshape_names) == 1:
        nodes.append(oh.make_node("Identity", [reshape_names[0]], ["output"]))
    else:
        prev = reshape_names[0]
        for i in range(1, len(reshape_names)):
            out = "output" if (i == len(reshape_names) - 1 and mask is None) else f"max_{i}"
            nodes.append(oh.make_node("Max", [prev, reshape_names[i]], [out]))
            prev = out
    if mask is not None:
        mask = _compact_mask(mask)
        nodes += [
            oh.make_node("Constant", [], ["mask"], value=onh.from_array(mask.astype(np.float32), name="mask")),
            oh.make_node("Mul", [prev, "mask"], ["output"]),
        ]
    graph = oh.make_graph(nodes, "sym_complete", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

# ============================================================
# DETECTORS
# ============================================================

def detect_rotation(pairs):
    for angle in [90, 180, 270]:
        k = angle // 90
        if all(np.rot90(np.array(p["input"]), k).shape == np.array(p["output"]).shape and
               np.array_equal(np.rot90(np.array(p["input"]), k), np.array(p["output"])) for p in pairs):
            return angle
    return None

def detect_flip(pairs):
    for axis_val, name in [(0, "vertical"), (1, "horizontal"), (-1, "both")]:
        def do_flip(ig, a=axis_val):
            return np.flip(np.flip(ig, 0), 1) if a == -1 else np.flip(ig, axis=a)
        if all(do_flip(np.array(p["input"])).shape == np.array(p["output"]).shape and
               np.array_equal(do_flip(np.array(p["input"])), np.array(p["output"])) for p in pairs):
            return name
    return None

def detect_transpose(pairs):
    return all(np.array(p["input"]).T.shape == np.array(p["output"]).shape and
               np.array_equal(np.array(p["input"]).T, np.array(p["output"])) for p in pairs)

def detect_anti_transpose(pairs):
    return all(np.flip(np.flip(np.array(p["input"]).T, 0), 1).shape == np.array(p["output"]).shape and
               np.array_equal(np.flip(np.flip(np.array(p["input"]).T, 0), 1), np.array(p["output"])) for p in pairs)

def detect_crop(pairs):
    results = []
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ > ih or ow_ > iw: return None
        found = False
        for r in range(ih - oh_ + 1):
            for c in range(iw - ow_ + 1):
                if np.array_equal(ig[r:r+oh_, c:c+ow_], og):
                    results.append((r, c, oh_, ow_)); found = True; break
            if found: break
        if not found: return None
    if len(set((r[0], r[1]) for r in results)) == 1:
        return results[0]
    return None

def detect_tile(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ % ih != 0 or ow_ % iw != 0: return None
        tr = oh_ // ih; tc = ow_ // iw
        if not np.array_equal(np.tile(ig, (tr, tc)), og): return None
    return (np.array(pairs[0]["output"]).shape[0] // np.array(pairs[0]["input"]).shape[0],
            np.array(pairs[0]["output"]).shape[1] // np.array(pairs[0]["input"]).shape[1])

def detect_scale(pairs):
    for factor in [2, 3, 4, 5]:
        if all(np.array(p["output"]).shape == (np.array(p["input"]).shape[0]*factor, np.array(p["input"]).shape[1]*factor) and
               np.array_equal(np.repeat(np.repeat(np.array(p["input"]), factor, 0), factor, 1), np.array(p["output"])) for p in pairs):
            return factor
    return None

def detect_color_replace(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    replacements = {}
    for p in pairs:
        for a, b in zip(np.array(p["input"]).flat, np.array(p["output"]).flat):
            if a != b:
                if a in replacements and replacements[a] != b: return None
                replacements[a] = b
    if not replacements: return None
    for p in pairs:
        ig = np.array(p["input"]); exp = ig.copy()
        for s, d in replacements.items(): exp[ig == s] = d
        if not np.array_equal(exp, np.array(p["output"])): return None
    return replacements

def detect_color_swap(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    swap_pairs = set()
    for p in pairs:
        for a, b in zip(np.array(p["input"]).flat, np.array(p["output"]).flat):
            if a != b: swap_pairs.add((min(a,b), max(a,b)))
    if len(swap_pairs) != 1: return None
    a, b = list(swap_pairs)[0]
    saw_ab = saw_ba = False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        exp = ig.copy(); exp[ig==a]=b; exp[ig==b]=a
        if not np.array_equal(exp, og): return None
        if np.any((ig==a)&(og==b)): saw_ab = True
        if np.any((ig==b)&(og==a)): saw_ba = True
    return (a,b) if saw_ab and saw_ba else None

def detect_pixel_permutation(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    if len(set(np.array(p["input"]).shape for p in pairs)) != 1: return None
    ih, iw = np.array(pairs[0]["input"]).shape
    if ih > H or iw > W: return None
    all_inputs = [np.array(p["input"]).flatten() for p in pairs]
    all_outputs = [np.array(p["output"]).flatten() for p in pairs]
    grid_size = ih * iw
    assignment = {}
    for dst_idx in range(grid_size):
        targets = [all_outputs[pi][dst_idx] for pi in range(len(pairs))]
        candidates = None
        for pi in range(len(pairs)):
            match = set(np.where(all_inputs[pi] == targets[pi])[0].tolist())
            candidates = match if candidates is None else candidates & match
            if not candidates: return None
        if len(candidates) != 1: return None
        assignment[dst_idx] = candidates.pop()
    gi = np.arange(HW, dtype=np.int64)
    for dst, src in assignment.items():
        dr, dc = divmod(dst, iw); sr, sc = divmod(src, iw)
        gi[dr*W+dc] = sr*W+sc
    for p in pairs:
        ig = np.array(p["input"]); padded = np.zeros((H, W), dtype=np.int32)
        padded[:ih, :iw] = ig
        pred = padded.flatten()[gi].reshape(H, W)[:ih, :iw]
        if not np.array_equal(pred, np.array(p["output"])): return None
    return gi

def detect_mirror_h_concat(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0], ig.shape[1]*2): return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 1)], 1), og): return False
    return True

def detect_mirror_v_concat(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0]*2, ig.shape[1]): return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 0)], 0), og): return False
    return True

def detect_self_concat_h(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0], ig.shape[1]*2): return False
        if not np.array_equal(np.concatenate([ig, ig], 1), og): return False
    return True

def detect_self_concat_v(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0]*2, ig.shape[1]): return False
        if not np.array_equal(np.concatenate([ig, ig], 0), og): return False
    return True

def detect_quad_mirror(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape
        if og.shape != (ih*2, iw*2): return None
        exp = np.block([[ig, np.flip(ig,1)], [np.flip(ig,0), np.flip(np.flip(ig,0),1)]])
        if np.array_equal(exp, og): continue
        return None
    return "tl"

def detect_h_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if not np.array_equal(np.where(ig != 0, ig, np.flip(ig, 1)), og): return False
    return True

def detect_v_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if not np.array_equal(np.where(ig != 0, ig, np.flip(ig, 0)), og): return False
    return True

def detect_both_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        stack = np.stack([ig, np.flip(ig,1), np.flip(ig,0), np.flip(np.flip(ig,0),1)], 0)
        if not np.array_equal(np.where(ig != 0, ig, stack.max(0)), og): return False
    return True

def detect_diag_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0] != ig.shape[1]: return False
        if not np.array_equal(np.where(ig != 0, ig, ig.T), og): return False
    return True

def detect_translation(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    ih, iw = np.array(pairs[0]["input"]).shape
    for dr in range(-min(ih,11)+1, min(ih,11)):
        for dc in range(-min(iw,11)+1, min(iw,11)):
            if dr == 0 and dc == 0: continue
            ok = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                hh, ww = ig.shape; s = np.zeros_like(ig)
                sr0=max(0,-dr); sr1=min(hh,hh-dr); sc0=max(0,-dc); sc1=min(ww,ww-dc)
                dr0=max(0,dr); dc0=max(0,dc); rh=sr1-sr0; rw=sc1-sc0
                if rh>0 and rw>0: s[dr0:dr0+rh,dc0:dc0+rw]=ig[sr0:sr1,sc0:sc1]
                if not np.array_equal(s, og): ok = False; break
            if ok: return (dr, dc)
    return None

def detect_downscale(pairs):
    for f in [2,3,4,5]:
        if all(np.array(p["input"]).shape == (np.array(p["output"]).shape[0]*f, np.array(p["output"]).shape[1]*f) and
               np.array_equal(np.array(p["input"])[::f,::f], np.array(p["output"])) for p in pairs):
            return f
    return None

def detect_color_filter(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    cand = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        diff = (ig != og)
        if not np.any(diff): continue
        if not np.all(og[diff] == 0): return None
        local = set(int(x) for x in ig[diff])
        cand = local if cand is None else cand | local
    if not cand: return None
    for p in pairs:
        ig = np.array(p["input"]).copy()
        for c in cand: ig[ig == c] = 0
        if not np.array_equal(ig, np.array(p["output"])): return None
    return cand

def detect_row_broadcast(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    chosen = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        m = None
        for r in range(ig.shape[0]):
            if np.all(og == ig[r, :]): m = r; break
        if m is None: return None
        if chosen is None: chosen = m
        elif chosen != m: return None
    return chosen

def detect_col_broadcast(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    chosen = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        m = None
        for c in range(ig.shape[1]):
            if np.all(og == ig[:, c:c+1]): m = c; break
        if m is None: return None
        if chosen is None: chosen = m
        elif chosen != m: return None
    return chosen

def detect_border_add(pairs):
    if not pairs: return None
    fi = np.array(pairs[0]["input"]); fo = np.array(pairs[0]["output"])
    ih, iw = fi.shape; oh_, ow_ = fo.shape
    if oh_-ih != ow_-iw or oh_ <= ih: return None
    d = oh_-ih
    if d % 2: return None
    b = d // 2
    if not np.array_equal(fo[b:b+ih, b:b+iw], fi): return None
    vals = set(fo[:b,:].flat) | set(fo[-b:,:].flat) | set(fo[:,:b].flat) | set(fo[:,-b:].flat)
    if len(vals) != 1: return None
    fill = int(vals.pop())
    for p in pairs[1:]:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0]+2*b, ig.shape[1]+2*b): return None
        if not np.array_equal(og[b:b+ig.shape[0], b:b+ig.shape[1]], ig): return None
        m = np.zeros(og.shape, dtype=bool); m[:b,:]=1; m[-b:,:]=1; m[:,:b]=1; m[:,-b:]=1
        if not np.all(og[m] == fill): return None
    return (b, fill)

def detect_crop_plus_color(pairs):
    if not pairs: return None
    fi = np.array(pairs[0]["input"]); fo = np.array(pairs[0]["output"])
    oh_, ow_ = fo.shape; ih, iw = fi.shape
    if oh_ > ih or ow_ > iw: return None
    for r0 in range(ih-oh_+1):
        for c0 in range(iw-ow_+1):
            cm = {}; ok = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                if og.shape != (oh_,ow_) or r0+oh_>ig.shape[0] or c0+ow_>ig.shape[1]: ok=False; break
                for a, b in zip(ig[r0:r0+oh_,c0:c0+ow_].flat, og.flat):
                    a,b=int(a),int(b)
                    if a in cm and cm[a]!=b: ok=False; break
                    cm[a]=b
                if not ok: break
            if not ok: continue
            if all(k==v for k,v in cm.items()): continue
            v = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                reg = ig[r0:r0+oh_,c0:c0+ow_].copy()
                for k2,v2 in cm.items(): reg[reg==k2]=v2
                if not np.array_equal(reg,og): v=False; break
            if v: return (r0,c0,oh_,ow_,cm)
    return None

def detect_concat_rot180_w(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if og.shape != (ig.shape[0], ig.shape[1]*2): return None
        if not np.array_equal(np.concatenate([ig, np.rot90(ig, 2)], 1), og): return None
    return True

def detect_nonzero_recolor(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    tc = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if not np.all(og[ig==0]==0): return None
        nb = og[ig!=0]
        if nb.size == 0: continue
        u = np.unique(nb)
        if len(u) != 1: return None
        if tc is None: tc = u[0]
        elif tc != u[0]: return None
    return tc

def detect_bg_recolor(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    tc = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if not np.array_equal(og[ig!=0], ig[ig!=0]): return None
        bo = og[ig==0]
        if bo.size == 0: continue
        u = np.unique(bo)
        if len(u) != 1: return None
        if tc is None: tc = u[0]
        elif tc != u[0]: return None
    return tc

# ============================================================
# SPATIAL SOLVERS (FIX #4: add relaxed version)
# ============================================================

def try_spatial_zero_ambiguity(all_pairs):
    for p in all_pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0]>H or ig.shape[1]>W or og.shape[0]>H or og.shape[1]>W: return None

    inputs = np.stack([grid_to_tensor(p["input"])[0].reshape(C, -1) for p in all_pairs])
    outputs = np.stack([grid_to_tensor(p["output"])[0].reshape(C, -1) for p in all_pairs])

    lookup = {}
    for si in range(HW):
        lookup.setdefault(inputs[:,:,si].tobytes(), []).append(si)

    gi = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for oi in range(HW):
        sa = outputs[:,:,oi]
        if np.all(sa == 0): continue
        cands = lookup.get(sa.tobytes())
        if not cands or len(cands) > 1: return None
        gi[oi] = cands[0]
    return make_gather_onnx(gi)

def try_spatial_relaxed(all_pairs, max_ambig=12):
    """FIX #4: Try spatial with ambiguity — brute-force combinations."""
    for p in all_pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0]>H or ig.shape[1]>W or og.shape[0]>H or og.shape[1]>W: return None

    inputs = np.stack([grid_to_tensor(p["input"])[0].reshape(C, -1) for p in all_pairs])
    outputs = np.stack([grid_to_tensor(p["output"])[0].reshape(C, -1) for p in all_pairs])

    lookup = {}
    for si in range(HW):
        lookup.setdefault(inputs[:,:,si].tobytes(), []).append(si)

    gi = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    ambig = []
    for oi in range(HW):
        sa = outputs[:,:,oi]
        if np.all(sa == 0): continue
        cands = lookup.get(sa.tobytes())
        if not cands: return None
        gi[oi] = cands[0]
        if len(cands) > 1: ambig.append((oi, cands))

    if len(ambig) > max_ambig: return None
    if not ambig:
        m = make_gather_onnx(gi)
        return m if check_model_correct(m, all_pairs) else None

    # Try default
    m = make_gather_onnx(gi)
    if check_model_correct(m, all_pairs): return m

    # Brute force (limit to 8 ambiguous positions)
    if len(ambig) > 8: return None
    for combo in iprod(*(c for _, c in ambig)):
        for idx, (oi, _) in enumerate(ambig): gi[oi] = combo[idx]
        m = make_gather_onnx(gi)
        if check_model_correct(m, all_pairs): return m
    return None

# ============================================================
# GATHER INDEX BUILDERS
# ============================================================

def build_rotation_gather(angle, ih, iw):
    k = angle // 90
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    if k == 1:
        for r in range(iw):
            for c in range(ih): idx[r*W+c] = c*W+(iw-1-r)
    elif k == 2:
        for r in range(ih):
            for c in range(iw): idx[r*W+c] = (ih-1-r)*W+(iw-1-c)
    elif k == 3:
        for r in range(iw):
            for c in range(ih): idx[r*W+c] = (ih-1-c)*W+r
    return idx

def build_flip_gather(direction, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            if direction == "vertical": idx[r*W+c] = (ih-1-r)*W+c
            elif direction == "horizontal": idx[r*W+c] = r*W+(iw-1-c)
            else: idx[r*W+c] = (ih-1-r)*W+(iw-1-c)
    return idx

def build_h_sym_complete_gathers(ih, iw):
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r*W+c] = r*W+c; idx_m[r*W+c] = r*W+(iw-1-c)
    return [idx_id, idx_m]

def build_v_sym_complete_gathers(ih, iw):
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r*W+c] = r*W+c; idx_m[r*W+c] = (ih-1-r)*W+c
    return [idx_id, idx_m]

def build_both_sym_complete_gathers(ih, iw):
    idxs = []
    for t in range(4):
        idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
        for r in range(ih):
            for c in range(iw):
                if t==0: sr,sc=r,c
                elif t==1: sr,sc=r,iw-1-c
                elif t==2: sr,sc=ih-1-r,c
                else: sr,sc=ih-1-r,iw-1-c
                idx[r*W+c] = sr*W+sc
        idxs.append(idx)
    return idxs

def build_diag_sym_complete_gathers(ih, iw):
    if ih != iw: return None
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r*W+c] = r*W+c; idx_m[r*W+c] = c*W+r
    return [idx_id, idx_m]

# ============================================================
# WEIGHT BUILDERS
# ============================================================

def can_use_channel_gather_for_cmap(cmap, train_pairs):
    if cmap is None: return False
    changed = {s: d for s, d in cmap.items() if s != d}
    if not changed: return False
    if set(changed.keys()) == set(changed.values()): return True
    return False

def build_channel_gather_indices(cmap):
    gi = np.arange(C, dtype=np.int32)
    for src, dst in cmap.items():
        if src != dst: gi[dst] = src
    return gi

def build_color_map_weight(color_map):
    w = np.zeros((C, C, 1, 1), dtype=np.float32)
    for src, dst in color_map.items():
        if 0 <= src < C and 0 <= dst < C: w[dst, src, 0, 0] = 1.0
    for ch in range(C):
        if ch not in color_map: w[ch, ch, 0, 0] = 1.0
    return w

def build_nonzero_recolor_weight(tc):
    w = np.zeros((C,C,1,1), dtype=np.float32); w[0,0,0,0]=1.0
    for s in range(1,C): w[tc,s,0,0]=1.0
    return w

def build_bg_recolor_weight(tc):
    w = np.zeros((C,C,1,1), dtype=np.float32); w[tc,0,0,0]=1.0
    for s in range(1,C): w[s,s,0,0]=1.0
    return w

# ============================================================
# LEARNED CONV (FIX #3: threshold 0.05, FIX #5: add bias, FIX #6: add 3-layer)
# ============================================================

def try_learned_conv_fast(pairs, all_pairs, kernel_size=1, max_steps=1500, lr=0.02, use_bias=False):
    try:
        import torch, torch.nn as nn
    except: return None

    inp = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    pad = kernel_size // 2
    conv = nn.Conv2d(C, C, kernel_size=kernel_size, padding=pad, bias=use_bias)
    nn.init.zeros_(conv.weight)
    if use_bias: nn.init.zeros_(conv.bias)
    for i in range(C): conv.weight.data[i, i, pad, pad] = 1.0

    optimizer = torch.optim.Adam(conv.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)
    best_loss = float("inf"); best_state = None; plateau = 0

    for step in range(max_steps):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(conv(inp), out)
        loss.backward(); optimizer.step(); scheduler.step()
        l = loss.item()
        if l < best_loss - 1e-7:
            best_loss = l; best_state = copy.deepcopy(conv.state_dict()); plateau = 0
        else: plateau += 1
        if step == 250 and best_loss > 0.3: return None
        if step == 600 and best_loss > 0.1: return None
        if plateau > 400 and best_loss > 0.01: return None
        if best_loss < 1e-6: break

    if best_loss > 0.05: return None  # FIX #3: was 0.005
    conv.load_state_dict(best_state)
    w = conv.weight.detach().numpy()
    b = conv.bias.detach().numpy() if use_bias else None

    # Try rounded, pruned, original
    best_model = None; best_cost = float('inf')
    for w_try in [np.round(w), np.where(np.abs(w)<0.01, 0, w), w]:
        b_try = np.round(b) if b is not None else None
        m = make_conv_onnx(w_try, bias=b_try, kernel_size=kernel_size)
        if check_model_correct(m, all_pairs):
            c = estimate_model_cost(m)
            if c < best_cost: best_cost = c; best_model = m
    return best_model

def try_two_layer_conv_fast(pairs, all_pairs, ks1=3, ks2=1, hidden=16, max_steps=2500, lr=0.015):
    try:
        import torch, torch.nn as nn
    except: return None

    inp = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    pad1 = ks1//2; pad2 = ks2//2
    torch.manual_seed(0)
    net = nn.Sequential(
        nn.Conv2d(C, hidden, kernel_size=ks1, padding=pad1, bias=True),
        nn.ReLU(),
        nn.Conv2d(hidden, C, kernel_size=ks2, padding=pad2, bias=False),
    )
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps)
    best_loss = float("inf"); best_state = None; plateau = 0

    for step in range(max_steps):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(net(inp), out)
        loss.backward(); optimizer.step(); scheduler.step()
        l = loss.item()
        if l < best_loss - 1e-7:
            best_loss = l; best_state = copy.deepcopy(net.state_dict()); plateau = 0
        else: plateau += 1
        if step == 300 and best_loss > 0.5: return None
        if step == 800 and best_loss > 0.15: return None
        if plateau > 500 and best_loss > 0.005: return None
        if best_loss < 1e-6: break

    if best_loss > 0.05: return None  # FIX #3: was 0.005
    net.load_state_dict(best_state)
    w1 = np.where(np.abs(v:=net[0].weight.detach().numpy())<0.01,0,v)
    b1 = net[0].bias.detach().numpy()
    w2 = np.where(np.abs(v:=net[2].weight.detach().numpy())<0.01,0,v)

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["w1"], value=onh.from_array(w1.astype(np.float32), name="w1")),
        oh.make_node("Constant", [], ["b1"], value=onh.from_array(b1.astype(np.float32), name="b1")),
        oh.make_node("Conv", ["input", "w1", "b1"], ["c1"], kernel_shape=[ks1,ks1], pads=[pad1]*4),
        oh.make_node("Relu", ["c1"], ["r1"]),
        oh.make_node("Constant", [], ["w2"], value=onh.from_array(w2.astype(np.float32), name="w2")),
        oh.make_node("Conv", ["r1", "w2"], ["output"], kernel_shape=[ks2,ks2], pads=[pad2]*4),
    ]
    graph = oh.make_graph(nodes, "two_layer", [X], [Y])
    m = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)]); m.ir_version = 8
    return m if check_model_correct(m, all_pairs) else None

def try_three_layer_conv(pairs, all_pairs, max_steps=3000):
    """FIX #6: 3-layer conv for harder tasks."""
    try:
        import torch, torch.nn as nn
    except: return None

    inp = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    class Net(nn.Module):
        def __init__(s):
            super().__init__()
            s.c1=nn.Conv2d(C,16,3,padding=1,bias=True)
            s.c2=nn.Conv2d(16,16,3,padding=1,bias=True)
            s.c3=nn.Conv2d(16,C,1,bias=True)
        def forward(s,x): return s.c3(torch.relu(s.c2(torch.relu(s.c1(x)))))

    torch.manual_seed(0)
    net = Net(); optimizer = torch.optim.Adam(net.parameters(), lr=0.005)
    best_loss = float("inf"); best_state = None
    for step in range(max_steps):
        optimizer.zero_grad()
        loss = nn.functional.mse_loss(net(inp), out)
        loss.backward(); optimizer.step()
        if loss.item() < best_loss:
            best_loss = loss.item(); best_state = copy.deepcopy(net.state_dict())
        if step == 500 and best_loss > 0.3: return None
        if best_loss < 1e-6: break
    if best_loss > 0.05: return None
    net.load_state_dict(best_state)

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = [
        oh.make_node("Constant", [], ["w1"], value=onh.from_array(net.c1.weight.detach().numpy().astype(np.float32), name="w1")),
        oh.make_node("Constant", [], ["b1"], value=onh.from_array(net.c1.bias.detach().numpy().astype(np.float32), name="b1")),
        oh.make_node("Conv", ["input","w1","b1"], ["c1"], kernel_shape=[3,3], pads=[1,1,1,1]),
        oh.make_node("Relu", ["c1"], ["r1"]),
        oh.make_node("Constant", [], ["w2"], value=onh.from_array(net.c2.weight.detach().numpy().astype(np.float32), name="w2")),
        oh.make_node("Constant", [], ["b2"], value=onh.from_array(net.c2.bias.detach().numpy().astype(np.float32), name="b2")),
        oh.make_node("Conv", ["r1","w2","b2"], ["c2"], kernel_shape=[3,3], pads=[1,1,1,1]),
        oh.make_node("Relu", ["c2"], ["r2"]),
        oh.make_node("Constant", [], ["w3"], value=onh.from_array(net.c3.weight.detach().numpy().astype(np.float32), name="w3")),
        oh.make_node("Constant", [], ["b3"], value=onh.from_array(net.c3.bias.detach().numpy().astype(np.float32), name="b3")),
        oh.make_node("Conv", ["r2","w3","b3"], ["output"], kernel_shape=[1,1], pads=[0,0,0,0]),
    ]
    graph = oh.make_graph(nodes, "three_layer", [X], [Y])
    m = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)]); m.ir_version = 8
    return m if check_model_correct(m, all_pairs) else None

# ============================================================
# SOLVER
# ============================================================
def analyze_transformation(pairs):
    info = {}
    in_sizes = [np.array(p["input"]).shape for p in pairs]
    out_sizes = [np.array(p["output"]).shape for p in pairs]
    info["in_sizes"] = in_sizes; info["out_sizes"] = out_sizes
    info["same_size"] = all(i==o for i,o in zip(in_sizes, out_sizes))
    info["const_output"] = len(set(json.dumps(p["output"]) for p in pairs)) == 1
    info["all_same_in_size"] = len(set(in_sizes)) == 1
    info["identity"] = info["same_size"] and all(np.array_equal(np.array(p["input"]),np.array(p["output"])) for p in pairs)
    if info["same_size"]:
        m = {}; valid = True
        for p in pairs:
            for a,b in zip(np.array(p["input"]).flat, np.array(p["output"]).flat):
                if a in m and m[a]!=b: valid=False; break
                m[a]=b
            if not valid: break
        info["color_map"] = m if valid and any(k!=v for k,v in m.items()) else None
    else:
        info["color_map"] = None
    return info

def solve_task(task_data, task_name="", time_budget=15.0):
    train_pairs = task_data.get("train", [])
    test_pairs = task_data.get("test", [])
    arc_gen = task_data.get("arc-gen", [])

    if not train_pairs: return make_identity_onnx()

    # FIX #2: validate on ALL pairs
    all_pairs = train_pairs + test_pairs + arc_gen

    info = analyze_transformation(train_pairs)
    best_model = None; best_cost = float('inf')
    t_start = time.time()

    def try_model(model, label=""):
        nonlocal best_model, best_cost
        if model is None: return False
        if check_model_correct(model, all_pairs):
            cost = estimate_model_cost(model)
            if cost < best_cost:
                best_cost = cost; best_model = model
            return True
        return False

    def elapsed(): return time.time() - t_start

    # ---- RULE-BASED ----
    if info.get("identity"):
        try_model(make_identity_onnx(), "identity")
        if best_model: return best_model

    # Color maps
    has_spatial = (detect_rotation(train_pairs) is not None or detect_flip(train_pairs) is not None or
                   detect_transpose(train_pairs) or detect_anti_transpose(train_pairs))

    if info.get("color_map") and info["same_size"]:
        cmap = info["color_map"]
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial:
            try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)), "cmap_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "cmap_conv")

    tc = detect_nonzero_recolor(train_pairs)
    if tc is not None: try_model(make_conv1x1_onnx(build_nonzero_recolor_weight(tc)), "nz_recolor")

    tc = detect_bg_recolor(train_pairs)
    if tc is not None: try_model(make_conv1x1_onnx(build_bg_recolor_weight(tc)), "bg_recolor")

    repl = detect_color_replace(train_pairs)
    if repl:
        cmap = {i:i for i in range(C)}; cmap.update(repl)
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial:
            try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)), "repl_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "repl_conv")

    sw = detect_color_swap(train_pairs)
    if sw and not has_spatial:
        a,b = sw; cmap = {i:i for i in range(C)}; cmap[a]=b; cmap[b]=a
        try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)), "swap_ch")

    # FIX #7: Color filter with conv1x1 fallback
    cf = detect_color_filter(train_pairs)
    if cf and info["same_size"]:
        cmap = {c_:0 for c_ in cf}; cmap.update({c_:c_ for c_ in range(C) if c_ not in cf})
        if can_use_channel_gather_for_cmap(cmap, train_pairs):
            try_model(make_channel_gather_onnx(build_channel_gather_indices(cmap)), "cf_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "cf_conv")  # FIX #7: always try conv fallback

    # Rotations
    rot = detect_rotation(train_pairs)
    if rot:
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_rotation_gather(rot, ih, iw)), f"rot_{rot}")
            if rot == 90: try_model(make_cheap_rot90ccw_framed(ih, iw), f"rot90_cheap")
            elif rot == 180: try_model(make_cheap_rot180_framed(ih, iw), f"rot180_cheap")
            elif rot == 270: try_model(make_cheap_rot90cw_framed(ih, iw), f"rot270_cheap")

    flip = detect_flip(train_pairs)
    if flip:
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_flip_gather(flip, ih, iw)), f"flip_{flip}")
            if flip == "vertical": try_model(make_cheap_flip_h_framed(ih, iw), "flip_v_cheap")
            elif flip == "horizontal": try_model(make_cheap_flip_w_framed(ih, iw), "flip_h_cheap")
            elif flip == "both": try_model(make_cheap_rot180_framed(ih, iw), "flip_b_cheap")

    if detect_transpose(train_pairs):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_cheap_transpose_framed(ih, iw), "tr_cheap")

    if detect_anti_transpose(train_pairs):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_cheap_anti_transpose_framed(ih, iw), "atr_cheap")

    # Composite: spatial + color
    if rot and info.get("color_map"):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi = build_rotation_gather(rot, ih, iw); cmap = info["color_map"]
            if can_use_channel_gather_for_cmap(cmap, train_pairs):
                try_model(make_spatial_then_channel_gather_onnx(gi, build_channel_gather_indices(cmap)), "rot_ch_cm")
            try_model(make_gather_then_conv1x1_onnx(gi, build_color_map_weight(cmap)), "rot_cm")

    if flip and info.get("color_map"):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi = build_flip_gather(flip, ih, iw); cmap = info["color_map"]
            if can_use_channel_gather_for_cmap(cmap, train_pairs):
                try_model(make_spatial_then_channel_gather_onnx(gi, build_channel_gather_indices(cmap)), "flip_ch_cm")
            try_model(make_gather_then_conv1x1_onnx(gi, build_color_map_weight(cmap)), "flip_cm")

    # Crop / tile / scale
    crop = detect_crop(train_pairs)
    if crop:
        r0,c0,oh_,ow_ = crop
        try_model(make_cheap_crop(r0,c0,oh_,ow_), "crop")

    tile = detect_tile(train_pairs)
    if tile:
        tr,tc = tile; ih,iw = np.array(train_pairs[0]["input"]).shape
        if ih*tr<=H and iw*tc<=W: try_model(make_cheap_tile(tr,tc,ih,iw), "tile")

    sc = detect_scale(train_pairs)
    if sc:
        ih,iw = np.array(train_pairs[0]["input"]).shape
        if ih*sc<=H and iw*sc<=W: try_model(make_cheap_upscale(sc,ih,iw), "scale")

    ds = detect_downscale(train_pairs)
    if ds:
        ih,iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_cheap_downscale(ds,ih,iw), "downscale")

    perm = detect_pixel_permutation(train_pairs)
    if perm is not None: try_model(make_gather_onnx(perm), "perm")

    # Mirror/concat patterns
    in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
    if len(in_shapes) == 1:
        ih,iw = list(in_shapes)[0]
        if detect_mirror_h_concat(train_pairs) and iw*2<=W: try_model(make_cheap_concat_flip_w(ih,iw), "mirror_h")
        if detect_mirror_v_concat(train_pairs) and ih*2<=H: try_model(make_cheap_concat_flip_h(ih,iw), "mirror_v")
        if detect_self_concat_h(train_pairs) and iw*2<=W: try_model(make_cheap_self_concat_w(ih,iw), "self_h")
        if detect_self_concat_v(train_pairs) and ih*2<=H: try_model(make_cheap_self_concat_h(ih,iw), "self_v")
        qm = detect_quad_mirror(train_pairs)
        if qm and ih*2<=H and iw*2<=W: try_model(make_cheap_quadrant_mirror(ih,iw), "quad")
        if detect_concat_rot180_w(train_pairs) and iw*2<=W: try_model(make_cheap_concat_rot180_w(ih,iw), "cat_r180")

    # Translation
    trans = detect_translation(train_pairs)
    if trans:
        dr,dc = trans
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes)==1:
            ih,iw = list(in_shapes)[0]
            m = make_cheap_framed_translation(ih,iw,dr,dc)
            if m: try_model(m, "translate")

    # Row/col broadcast
    in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
    if len(in_shapes)==1:
        ih,iw = list(in_shapes)[0]
        rb = detect_row_broadcast(train_pairs)
        if rb is not None: try_model(make_cheap_row_broadcast(ih,iw,rb), "rb")
        cb = detect_col_broadcast(train_pairs)
        if cb is not None: try_model(make_cheap_col_broadcast(ih,iw,cb), "cb")

    # Border add
    ba = detect_border_add(train_pairs)
    if ba:
        b,fill = ba
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes)==1:
            ih,iw = list(in_shapes)[0]
            if ih+2*b<=H and iw+2*b<=W:
                m = make_cheap_border_add(ih,iw,b,fill)
                if m: try_model(m, "border")

    # Crop + color
    ccp = detect_crop_plus_color(train_pairs)
    if ccp:
        r0,c0,oh_,ow_,cm = ccp
        try_model(make_cheap_crop_then_color(r0,c0,oh_,ow_,cm), "crop_color")

    # Symmetry completion
    if info["same_size"] and info["all_same_in_size"]:
        ih,iw = np.array(train_pairs[0]["input"]).shape
        if detect_h_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_h_sym_complete_gathers(ih,iw)), "h_sym")
        if detect_v_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_v_sym_complete_gathers(ih,iw)), "v_sym")
        if detect_both_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_both_sym_complete_gathers(ih,iw)), "both_sym")
        if ih==iw and detect_diag_symmetry_complete(train_pairs):
            g = build_diag_sym_complete_gathers(ih,iw)
            if g: try_model(make_symmetry_completion_onnx(g), "diag_sym")

    # Spatial solvers
    m = try_spatial_zero_ambiguity(all_pairs)
    if m: try_model(m, "spatial_0")

    if best_model is None and elapsed() < time_budget:
        m = try_spatial_relaxed(all_pairs, max_ambig=12)
        if m: try_model(m, "spatial_r")

    if best_model is not None:
        return best_model

    # ---- LEARNED CONV ----
    for ks in [1, 3, 5]:
        if elapsed() > time_budget: break
        m = try_learned_conv_fast(train_pairs, all_pairs, kernel_size=ks, use_bias=False)
        if try_model(m, f"lconv_{ks}"): break
        m = try_learned_conv_fast(train_pairs, all_pairs, kernel_size=ks, use_bias=True)
        if try_model(m, f"lconv_{ks}b"): break

    if best_model is not None: return best_model

    for ks1,ks2,hidden in [(3,1,16),(3,1,24),(3,1,8),(5,1,12)]:
        if elapsed() > time_budget: break
        m = try_two_layer_conv_fast(train_pairs, all_pairs, ks1=ks1, ks2=ks2, hidden=hidden)
        if try_model(m, f"l2_{ks1}_{hidden}"): break

    if best_model is not None: return best_model

    if elapsed() < time_budget:
        m = try_three_layer_conv(train_pairs, all_pairs)
        try_model(m, "l3")

    if best_model is not None: return best_model

    # Constant output (last resort)
    if info.get("const_output"):
        try_model(make_const_onnx(grid_to_tensor(train_pairs[0]["output"])), "const")

    return best_model

# ============================================================
# MAIN
# ============================================================
def save_model(model, path):
    with open(path, "wb") as f:
        f.write(model.SerializeToString())

In [ ]:
%%writefile fast_solver.py

import os, json, math, zipfile, copy, itertools, sys
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any
import numpy as np

try:
    import onnx, onnx.helper as oh, onnx.numpy_helper as onh
    from onnx import TensorProto
except ImportError:
    os.system("pip install onnx -q")
    import onnx, onnx.helper as oh, onnx.numpy_helper as onh
    from onnx import TensorProto
try:
    import onnxruntime as ort
except ImportError:
    os.system("pip install onnxruntime -q")
    import onnxruntime as ort

C,H,W=10,30,30
MAX_BYTES=int(1.44*1024*1024)
BAD_OPS={"Loop","Scan","NonZero","Unique","Script","Function"}
OPSET=[oh.make_opsetid("",17)]

def grid_to_tensor(grid):
    g=np.asarray(grid,dtype=np.int32); h,w=g.shape
    t=np.zeros((1,C,H,W),dtype=np.float32)
    for r in range(h):
        for c in range(w):
            v=int(g[r,c])
            if 0<=v<C: t[0,v,r,c]=1.0
    return t

def tensor_to_grid(t,oh_,ow_):
    arr=t[0,:,:oh_,:ow_]; grid=[]
    for r in range(oh_):
        row=[]
        for c in range(ow_):
            cv=arr[:,r,c]
            row.append(0 if cv.max()<0.5 else int(np.argmax(cv)))
        grid.append(row)
    return grid

def _X(): return oh.make_tensor_value_info("input",TensorProto.FLOAT,[1,C,H,W])
def _Y(): return oh.make_tensor_value_info("output",TensorProto.FLOAT,[1,C,H,W])
def _c(name,arr): return oh.make_node("Constant",[],[name],value=onh.from_array(arr,name=name+"_v"))
def _m(nodes,gn="net"):
    m=oh.make_model(oh.make_graph(nodes,gn,[_X()],[_Y()]),opset_imports=OPSET); m.ir_version=8; return m

def model_bytes(m): return m.SerializeToString()
def check_fits(m): return len(model_bytes(m))<=MAX_BYTES
def check_ops(m): return all(n.op_type not in BAD_OPS for n in m.graph.node)

def count_params(m):
    t=0
    for i in m.graph.initializer: t+=int(np.prod(i.dims)) if i.dims else 0
    for n in m.graph.node:
        if n.op_type=="Constant":
            for a in n.attribute:
                if a.name=="value" and a.t.dims: t+=int(np.prod(a.t.dims))
    return t

def _smap(m):
    s={}
    for vi in list(m.graph.input)+list(m.graph.output):
        s[vi.name]=tuple(d.dim_value if d.dim_value>0 else 1 for d in vi.type.tensor_type.shape.dim)
    for i in m.graph.initializer: s[i.name]=tuple(i.dims)
    for n in m.graph.node:
        if n.op_type=="Constant":
            for a in n.attribute:
                if a.name=="value": s[n.output[0]]=tuple(a.t.dims) if a.t.dims else ()
    return s

def est_macs(m):
    mc=0; s=_smap(m)
    for n in m.graph.node:
        if n.op_type=="Conv":
            w=s.get(n.input[1]); x=s.get(n.input[0])
            if w and x and len(w)==4 and len(x)==4: mc+=int(w[0]*w[1]*w[2]*w[3]*x[2]*x[3])
        elif n.op_type=="Gemm":
            a=s.get(n.input[0]); w=s.get(n.input[1])
            if a and w and len(a)>=2 and len(w)>=2: mc+=int(a[0]*a[1]*(w[0] if w[0]!=a[1] else w[1]))
        elif n.op_type=="MatMul":
            a=s.get(n.input[0]); b=s.get(n.input[1])
            if a and b and len(a)>=2 and len(b)>=2: mc+=int(a[-2]*a[-1]*b[-1])
    return mc

def cost(m): return count_params(m)+len(model_bytes(m))+est_macs(m)

def check_correct(m,pairs):
    if not pairs or not check_fits(m) or not check_ops(m): return False
    try:
        so=ort.SessionOptions(); so.log_severity_level=3
        sess=ort.InferenceSession(model_bytes(m),sess_options=so,providers=["CPUExecutionProvider"])
    except: return False
    try:
        for p in pairs:
            inp=grid_to_tensor(p["input"]); og=np.asarray(p["output"]); oh_,ow_=og.shape
            pred=sess.run(None,{"input":inp})[0]
            if tensor_to_grid(pred,oh_,ow_)!=p["output"]: return False
        return True
    except: return False

# ═══════════════════════════════════════════════════════════
# Builders
# ═══════════════════════════════════════════════════════════

def mk_identity(): return _m([oh.make_node("Identity",["input"],["output"])],"id")

def mk_const(og):
    og=np.asarray(og,dtype=np.int32); oh_,ow_=og.shape
    p=np.zeros((1,C,H,W),dtype=np.float32)
    for r in range(oh_):
        for c in range(ow_):
            v=int(og[r,c])
            if 0<=v<C: p[0,v,r,c]=1.0
    return _m([_c("K",p),oh.make_node("Identity",["K"],["output"])],"const")

def mk_transpose(): return _m([oh.make_node("Transpose",["input"],["output"],perm=[0,1,3,2])],"tr")

def _sr(ax,dm,i,o,s=""):
    return [_c(f"s{s}",np.asarray([d-1 for d in dm],dtype=np.int64)),
            _c(f"e{s}",np.asarray([-d-1 for d in dm],dtype=np.int64)),
            _c(f"a{s}",np.asarray(ax,dtype=np.int64)),
            _c(f"p{s}",np.asarray([-1]*len(dm),dtype=np.int64)),
            oh.make_node("Slice",[i,f"s{s}",f"e{s}",f"a{s}",f"p{s}"],[o])]

def mk_flip_h(): return _m(_sr([2],[H],"input","output"),"fh")
def mk_flip_w(): return _m(_sr([3],[W],"input","output"),"fw")
def mk_flip_both(): return _m(_sr([2,3],[H,W],"input","output"),"fb")
def mk_rot180(): return mk_flip_both()
def mk_rot90cw():
    n=[oh.make_node("Transpose",["input"],["t"],perm=[0,1,3,2])]; n+=_sr([3],[W],"t","output","r"); return _m(n,"r90cw")
def mk_rot90ccw():
    n=[oh.make_node("Transpose",["input"],["t"],perm=[0,1,3,2])]; n+=_sr([2],[H],"t","output","r"); return _m(n,"r90ccw")
def mk_anti_tr():
    n=[oh.make_node("Transpose",["input"],["t"],perm=[0,1,3,2])]; n+=_sr([2,3],[H,W],"t","output","r"); return _m(n,"atr")

def _pad(i,o,oh_,ow_,s=""):
    return [_c(f"pd{s}",np.asarray([0,0,0,0,0,0,H-oh_,W-ow_],dtype=np.int64)),
            oh.make_node("Pad",[i,f"pd{s}"],[o],mode="constant")]

def _crop(i,o,h,w,s=""):
    return [_c(f"cs{s}",np.asarray([0,0],dtype=np.int64)),
            _c(f"ce{s}",np.asarray([h,w],dtype=np.int64)),
            _c(f"ca{s}",np.asarray([2,3],dtype=np.int64)),
            oh.make_node("Slice",[i,f"cs{s}",f"ce{s}",f"ca{s}"],[o])]

def mkf_rot180(h,w): n=_crop("input","ct",h,w,"0"); n+=_sr([2,3],[h,w],"ct","rt","r"); n+=_pad("rt","output",h,w,"p"); return _m(n,"fr180")
def mkf_fh(h,w): n=_crop("input","ct",h,w,"0"); n+=_sr([2],[h],"ct","fl","r"); n+=_pad("fl","output",h,w,"p"); return _m(n,"ffh")
def mkf_fw(h,w): n=_crop("input","ct",h,w,"0"); n+=_sr([3],[w],"ct","fl","r"); n+=_pad("fl","output",h,w,"p"); return _m(n,"ffw")
def mkf_tr(h,w): n=_crop("input","ct",h,w,"0"); n+=[oh.make_node("Transpose",["ct"],["tp"],perm=[0,1,3,2])]; n+=_pad("tp","output",w,h,"p"); return _m(n,"ftr")
def mkf_r90cw(h,w): n=_crop("input","ct",h,w,"0"); n+=[oh.make_node("Transpose",["ct"],["t"],perm=[0,1,3,2])]; n+=_sr([3],[h],"t","rt","r"); n+=_pad("rt","output",w,h,"p"); return _m(n,"fr90cw")
def mkf_r90ccw(h,w): n=_crop("input","ct",h,w,"0"); n+=[oh.make_node("Transpose",["ct"],["t"],perm=[0,1,3,2])]; n+=_sr([2],[w],"t","rt","r"); n+=_pad("rt","output",w,h,"p"); return _m(n,"fr90ccw")

def mk_color_remap(cm):
    inv={d:s for s,d in cm.items()}; idx=np.arange(C,dtype=np.int64); mask=np.ones(C,dtype=np.float32)
    sa={s for s,d in cm.items() if s!=d}
    for d in range(C):
        if d in inv: idx[d]=inv[d]
        elif d in sa: idx[d]=0; mask[d]=0.0
    gi=_c("gi",idx)
    if mask.min()==1.0: return _m([gi,oh.make_node("Gather",["input","gi"],["output"],axis=1)],"crm")
    return _m([gi,oh.make_node("Gather",["input","gi"],["g"],axis=1),_c("cm",mask.reshape(1,C,1,1)),oh.make_node("Mul",["g","cm"],["output"])],"crm_m")

def mk_crop_slice(r0,c0,oh_,ow_):
    n=[_c("cs",np.asarray([r0,c0],dtype=np.int64)),_c("ce",np.asarray([r0+oh_,c0+ow_],dtype=np.int64)),
       _c("ca",np.asarray([2,3],dtype=np.int64)),oh.make_node("Slice",["input","cs","ce","ca"],["cr"])]
    n+=_pad("cr","output",oh_,ow_,"p"); return _m(n,"crop")

def mk_ftranslate(gh,gw,dr,dc):
    sr0=max(0,-dr);sr1=min(gh,gh-dr);sc0=max(0,-dc);sc1=min(gw,gw-dc)
    dr0=max(0,dr);dc0=max(0,dc);rh=sr1-sr0;rw=sc1-sc0
    if rh<=0 or rw<=0: return mk_const(np.zeros((1,1),dtype=np.int32))
    n=[_c("fts",np.asarray([sr0,sc0],dtype=np.int64)),_c("fte",np.asarray([sr1,sc1],dtype=np.int64)),
       _c("fta",np.asarray([2,3],dtype=np.int64)),oh.make_node("Slice",["input","fts","fte","fta"],["fs"])]
    n+=[_c("fpp",np.asarray([0,0,dr0,dc0,0,0,H-dr0-rh,W-dc0-rw],dtype=np.int64)),
        oh.make_node("Pad",["fs","fpp"],["output"],mode="constant")]
    return _m(n,"ftr")

def mk_upscale(f,ih,iw):
    n=_crop("input","ct",ih,iw,"0")
    n+=[_c("rr",np.asarray([],dtype=np.float32)),_c("rs",np.asarray([],dtype=np.float32)),
        _c("rz",np.asarray([1,C,ih*f,iw*f],dtype=np.int64)),
        oh.make_node("Resize",["ct","rr","rs","rz"],["up"],mode="nearest",
                     coordinate_transformation_mode="asymmetric",nearest_mode="floor")]
    n+=_pad("up","output",ih*f,iw*f,"p"); return _m(n,"upscale")

def mk_downscale(f,ih,iw):
    oh_=ih//f;ow_=iw//f
    n=[_c("ds",np.asarray([0,0],dtype=np.int64)),_c("de",np.asarray([ih,iw],dtype=np.int64)),
       _c("da",np.asarray([2,3],dtype=np.int64)),_c("dp",np.asarray([f,f],dtype=np.int64)),
       oh.make_node("Slice",["input","ds","de","da","dp"],["dn"])]
    n+=_pad("dn","output",oh_,ow_,"p"); return _m(n,"downscale")

def mk_tile(tr,tc,ih,iw):
    n=_crop("input","ct",ih,iw,"0")
    n+=[_c("tr",np.asarray([1,1,tr,tc],dtype=np.int64)),oh.make_node("Tile",["ct","tr"],["tl"])]
    n+=_pad("tl","output",ih*tr,iw*tc,"p"); return _m(n,"tile")

def mk_cat_fw(ih,iw):
    n=_crop("input","ct",ih,iw,"0"); n+=_sr([3],[iw],"ct","fl","fw")
    n+=[oh.make_node("Concat",["ct","fl"],["cat"],axis=3)]; n+=_pad("cat","output",ih,iw*2,"p"); return _m(n,"catfw")

def mk_cat_fh(ih,iw):
    n=_crop("input","ct",ih,iw,"0"); n+=_sr([2],[ih],"ct","fl","fh")
    n+=[oh.make_node("Concat",["ct","fl"],["cat"],axis=2)]; n+=_pad("cat","output",ih*2,iw,"p"); return _m(n,"catfh")

def mk_quad(ih,iw):
    n=_crop("input","ct",ih,iw,"0")
    n+=_sr([3],[iw],"ct","fw","fw"); n+=[oh.make_node("Concat",["ct","fw"],["top"],axis=3)]
    n+=_sr([2],[ih],"ct","fh","fh"); n+=_sr([2,3],[ih,iw],"ct","fr","fr")
    n+=[oh.make_node("Concat",["fh","fr"],["bot"],axis=3),oh.make_node("Concat",["top","bot"],["full"],axis=2)]
    n+=_pad("full","output",ih*2,iw*2,"p"); return _m(n,"quad")

def mk_row_bc(ih,iw,ri):
    n=[_c("rs",np.asarray([ri],dtype=np.int64)),_c("re",np.asarray([ri+1],dtype=np.int64)),_c("ra",np.asarray([2],dtype=np.int64)),
       oh.make_node("Slice",["input","rs","re","ra"],["row"]),
       _c("rr",np.asarray([1,1,ih,1],dtype=np.int64)),oh.make_node("Tile",["row","rr"],["tl"]),
       _c("rcs",np.asarray([0,0],dtype=np.int64)),_c("rce",np.asarray([ih,iw],dtype=np.int64)),_c("rca",np.asarray([2,3],dtype=np.int64)),
       oh.make_node("Slice",["tl","rcs","rce","rca"],["cr"])]
    n+=_pad("cr","output",ih,iw,"p"); return _m(n,"rowbc")

def mk_col_bc(ih,iw,ci):
    n=[_c("cs2",np.asarray([ci],dtype=np.int64)),_c("ce2",np.asarray([ci+1],dtype=np.int64)),_c("ca2",np.asarray([3],dtype=np.int64)),
       oh.make_node("Slice",["input","cs2","ce2","ca2"],["col"]),
       _c("cr2",np.asarray([1,1,1,iw],dtype=np.int64)),oh.make_node("Tile",["col","cr2"],["tl"]),
       _c("rcs",np.asarray([0,0],dtype=np.int64)),_c("rce",np.asarray([ih,iw],dtype=np.int64)),_c("rca",np.asarray([2,3],dtype=np.int64)),
       oh.make_node("Slice",["tl","rcs","rce","rca"],["cr"])]
    n+=_pad("cr","output",ih,iw,"p"); return _m(n,"colbc")

def mk_crop_color(r0,c0,oh_,ow_,cm):
    n=[_c("cs",np.asarray([r0,c0],dtype=np.int64)),_c("ce",np.asarray([r0+oh_,c0+ow_],dtype=np.int64)),
       _c("ca",np.asarray([2,3],dtype=np.int64)),oh.make_node("Slice",["input","cs","ce","ca"],["cr"])]
    inv={d:s for s,d in cm.items()}; idx=np.arange(C,dtype=np.int64); mask=np.ones(C,dtype=np.float32)
    sa={s for s,d in cm.items() if s!=d}
    for d in range(C):
        if d in inv: idx[d]=inv[d]
        elif d in sa: idx[d]=0; mask[d]=0.0
    gi=_c("gi",idx)
    if mask.min()==1.0: n+=[gi,oh.make_node("Gather",["cr","gi"],["cl"],axis=1)]
    else: n+=[gi,oh.make_node("Gather",["cr","gi"],["g"],axis=1),_c("cm",mask.reshape(1,C,1,1)),oh.make_node("Mul",["g","cm"],["cl"])]
    n+=_pad("cl","output",oh_,ow_,"p"); return _m(n,"cropcol")

def mk_border(gh,gw,b,fill):
    oh_=gh+2*b; ow_=gw+2*b
    if oh_>H or ow_>W: return None
    n=_crop("input","ct",gh,gw,"pb")
    n+=[_c("pbp",np.asarray([0,0,b,b,0,0,b,b],dtype=np.int64)),oh.make_node("Pad",["ct","pbp"],["pd"],mode="constant")]
    mask=np.zeros((1,C,oh_,ow_),dtype=np.float32)
    if 0<=fill<C: mask[0,fill,:b,:]=1;mask[0,fill,-b:,:]=1;mask[0,fill,:,:b]=1;mask[0,fill,:,-b:]=1
    n+=[_c("pbm",mask),oh.make_node("Add",["pd","pbm"],["bd"])]
    n+=_pad("bd","output",oh_,ow_,"pf"); return _m(n,"border")

_ZM=np.ones((1,C,1,1),dtype=np.float32); _ZM[0,0,0,0]=0.0
def _z0(i,o,s="z"): return [_c(f"{s}m",_ZM.copy()),oh.make_node("Mul",[i,f"{s}m"],[o])]

def mk_sym_h(ih,iw):
    n=_crop("input","ct",ih,iw,"0"); n+=_z0("ct","nz","sh"); n+=_sr([2],[ih],"nz","fl","shr")
    n+=[oh.make_node("Max",["nz","fl"],["cb"])]; n+=_pad("cb","output",ih,iw,"p"); return _m(n,"symh")
def mk_sym_w(ih,iw):
    n=_crop("input","ct",ih,iw,"0"); n+=_z0("ct","nz","sw"); n+=_sr([3],[iw],"nz","fl","swr")
    n+=[oh.make_node("Max",["nz","fl"],["cb"])]; n+=_pad("cb","output",ih,iw,"p"); return _m(n,"symw")
def mk_sym_hw(ih,iw):
    n=_crop("input","ct",ih,iw,"0"); n+=_z0("ct","nz","sx")
    n+=_sr([2],[ih],"nz","fh","sxh"); n+=_sr([3],[iw],"nz","fw","sxw"); n+=_sr([2,3],[ih,iw],"nz","fb","sxb")
    n+=[oh.make_node("Max",["nz","fh"],["m1"]),oh.make_node("Max",["m1","fw"],["m2"]),oh.make_node("Max",["m2","fb"],["cb"])]
    n+=_pad("cb","output",ih,iw,"p"); return _m(n,"symhw")
def mk_sym_diag(ih,iw):
    if ih!=iw: return None
    n=_crop("input","ct",ih,iw,"0"); n+=_z0("ct","nz","sd")
    n+=[oh.make_node("Transpose",["nz"],["tr"],perm=[0,1,3,2]),oh.make_node("Max",["nz","tr"],["cb"])]
    n+=_pad("cb","output",ih,iw,"p"); return _m(n,"symd")

def mk_xform_color(base_fn,cm):
    base=base_fn(); bn=[copy.deepcopy(n) for n in base.graph.node]
    for n in bn:
        for i,o in enumerate(n.output):
            if o=="output": n.output[i]="mid"
    inv={d:s for s,d in cm.items()}; idx=np.arange(C,dtype=np.int64); mask=np.ones(C,dtype=np.float32)
    sa={s for s,d in cm.items() if s!=d}
    for d in range(C):
        if d in inv: idx[d]=inv[d]
        elif d in sa: idx[d]=0; mask[d]=0.0
    gi=_c("gi2",idx)
    if mask.min()==1.0: return _m(bn+[gi,oh.make_node("Gather",["mid","gi2"],["output"],axis=1)],"xc")
    return _m(bn+[gi,oh.make_node("Gather",["mid","gi2"],["g2"],axis=1),_c("cm2",mask.reshape(1,C,1,1)),oh.make_node("Mul",["g2","cm2"],["output"])],"xc_m")

def mk_conv(w_np,ks,gn="conv"):
    pad=ks//2
    return _m([oh.make_node("Constant",[],["cw"],value=onh.from_array(w_np.astype(np.float32),name="cw")),
               oh.make_node("Conv",["input","cw"],["output"],kernel_shape=[ks,ks],pads=[pad]*4)],gn)

def mk_2layer(w1,b1,w2,b2):
    return _m([oh.make_node("Constant",[],["w1"],value=onh.from_array(w1,name="w1")),
               oh.make_node("Constant",[],["b1"],value=onh.from_array(b1,name="b1")),
               oh.make_node("Conv",["input","w1","b1"],["c1"],kernel_shape=[3,3],pads=[1,1,1,1]),
               oh.make_node("Relu",["c1"],["r1"]),
               oh.make_node("Constant",[],["w2"],value=onh.from_array(w2,name="w2")),
               oh.make_node("Constant",[],["b2"],value=onh.from_array(b2,name="b2")),
               oh.make_node("Conv",["r1","w2","b2"],["output"],kernel_shape=[1,1],pads=[0,0,0,0])],"2l")

# ═══════════════════════════════════════════════════════════
# Spatial copy solver
# ═══════════════════════════════════════════════════════════

def mk_spatial(gi,mask):
    g=oh.make_graph(
        [oh.make_node("Reshape",["input","shc"],["flat"]),
         oh.make_node("Gather",["flat","gi"],["fg"],axis=2),
         oh.make_node("Reshape",["fg","sho"],["gd"]),
         oh.make_node("Mul",["gd","mk"],["output"])],
        "spatial",[oh.make_tensor_value_info("input",TensorProto.FLOAT,[1,10,30,30])],
        [oh.make_tensor_value_info("output",TensorProto.FLOAT,[1,10,30,30])],
        [onh.from_array(np.array([1,10,900],dtype=np.int64),name="shc"),
         onh.from_array(np.array([1,10,30,30],dtype=np.int64),name="sho"),
         onh.from_array(gi.astype(np.int64),name="gi"),
         onh.from_array(mask.astype(np.float32),name="mk")])
    m=oh.make_model(g,opset_imports=OPSET); m.ir_version=8; return m

def solve_spatial(task_data):
    pairs=task_data.get("train",[])+task_data.get("test",[])+task_data.get("arc-gen",[])
    for p in pairs:
        if len(p["input"])>30 or len(p["input"][0])>30 or len(p["output"])>30 or len(p["output"][0])>30: return None
    inputs=np.stack([grid_to_tensor(p["input"])[0].reshape(10,-1) for p in pairs])
    outputs=np.stack([grid_to_tensor(p["output"])[0].reshape(10,-1) for p in pairs])
    lookup={}
    for si in range(900):
        sig=inputs[:,:,si].tobytes()
        lookup.setdefault(sig,[]).append(si)
    gi=np.zeros(900,dtype=np.int64); mask=np.zeros((1,1,30,30),dtype=np.float32)
    for oi in range(900):
        sa=outputs[:,:,oi]
        if np.all(sa==0.0): continue
        cands=lookup.get(sa.tobytes())
        if not cands or len(cands)>1: return None
        gi[oi]=cands[0]; r,c=divmod(oi,30); mask[0,0,r,c]=1.0
    m=mk_spatial(gi,mask)
    if check_correct(m,pairs): return m
    return None

# ═══════════════════════════════════════════════════════════
# Learned conv
# ═══════════════════════════════════════════════════════════

def try_conv(pairs,ks=1,steps=800):
    try: import torch, torch.nn as nn
    except: return None
    if not all(np.asarray(p["input"]).shape==np.asarray(p["output"]).shape for p in pairs): return None
    it=torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]),dtype=torch.float32)
    ot=torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]),dtype=torch.float32)
    pad=ks//2; conv=nn.Conv2d(C,C,ks,padding=pad,bias=False); nn.init.zeros_(conv.weight)
    opt=torch.optim.Adam(conv.parameters(),lr=0.05)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,patience=50,factor=0.5)
    bl=float("inf"); bs=None
    for _ in range(steps):
        opt.zero_grad(); l=nn.functional.mse_loss(conv(it),ot); l.backward(); opt.step(); sch.step(l.item())
        if l.item()<bl: bl=l.item(); bs=copy.deepcopy(conv.state_dict())
    if bl>0.05: return None
    conv.load_state_dict(bs)
    return mk_conv(conv.weight.detach().numpy(),ks,f"lconv{ks}")

def try_2layer(pairs):
    try: import torch, torch.nn as nn
    except: return None
    if not all(np.asarray(p["input"]).shape==np.asarray(p["output"]).shape for p in pairs): return None
    it=torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]),dtype=torch.float32)
    ot=torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]),dtype=torch.float32)
    class Net(nn.Module):
        def __init__(self): super().__init__(); self.c1=nn.Conv2d(C,32,3,padding=1,bias=True); self.c2=nn.Conv2d(32,C,1,bias=True)
        def forward(self,x): return self.c2(torch.relu(self.c1(x)))
    net=Net(); opt=torch.optim.Adam(net.parameters(),lr=0.01); bl=float("inf"); bs=None
    for _ in range(2000):
        opt.zero_grad(); l=nn.functional.mse_loss(net(it),ot); l.backward(); opt.step()
        if l.item()<bl: bl=l.item(); bs=copy.deepcopy(net.state_dict())
        if bl<0.01: break
    if bl>0.05: return None
    net.load_state_dict(bs)
    return mk_2layer(net.c1.weight.detach().numpy().astype(np.float32),net.c1.bias.detach().numpy().astype(np.float32),
                     net.c2.weight.detach().numpy().astype(np.float32),net.c2.bias.detach().numpy().astype(np.float32))

# ═══════════════════════════════════════════════════════════
# Detectors
# ═══════════════════════════════════════════════════════════

def all_same(p): return all(np.asarray(x["input"]).shape==np.asarray(x["output"]).shape for x in p)
def fsize(p):
    if not p: return None
    f=np.asarray(p[0]["input"]).shape
    return f if all(np.asarray(x["input"]).shape==f for x in p[1:]) else None

def d_id(p): return all(np.array_equal(np.asarray(x["input"]),np.asarray(x["output"])) for x in p)
def d_const(p):
    if not p: return None
    f=np.asarray(p[0]["output"])
    for x in p[1:]:
        o=np.asarray(x["output"])
        if o.shape!=f.shape or not np.array_equal(o,f): return None
    if len({str(x["input"]) for x in p})<2 and d_id(p): return None
    return f

def d_rot(p):
    for l,k in [("180",2),("90cw",-1),("90ccw",1)]:
        if all(np.rot90(np.asarray(x["input"]),k).shape==np.asarray(x["output"]).shape and
               np.array_equal(np.rot90(np.asarray(x["input"]),k),np.asarray(x["output"])) for x in p): return l
    return None

def d_flip(p):
    for n,fn in [("h",np.flipud),("w",np.fliplr),("both",lambda a:np.flipud(np.fliplr(a)))]:
        if all(fn(np.asarray(x["input"])).shape==np.asarray(x["output"]).shape and
               np.array_equal(fn(np.asarray(x["input"])),np.asarray(x["output"])) for x in p): return n
    return None

def d_tr(p):
    return all(np.asarray(x["input"]).T.shape==np.asarray(x["output"]).shape and
               np.array_equal(np.asarray(x["input"]).T,np.asarray(x["output"])) for x in p)

def d_atr(p):
    return all(np.flipud(np.fliplr(np.asarray(x["input"]).T)).shape==np.asarray(x["output"]).shape and
               np.array_equal(np.flipud(np.fliplr(np.asarray(x["input"]).T)),np.asarray(x["output"])) for x in p)

def d_cm(p):
    if not all_same(p): return None
    m={}
    for x in p:
        for a,b in zip(np.asarray(x["input"]).flat,np.asarray(x["output"]).flat):
            a,b=int(a),int(b)
            if a in m and m[a]!=b: return None
            m[a]=b
    return m if any(k!=v for k,v in m.items()) else None

def d_xc(p,fn):
    m={}
    for x in p:
        ig=np.asarray(x["input"]); og=np.asarray(x["output"]); t=fn(ig)
        if t.shape!=og.shape: return None
        for a,b in zip(t.flat,og.flat):
            a,b=int(a),int(b)
            if a in m and m[a]!=b: return None
            m[a]=b
    return m if any(k!=v for k,v in m.items()) else None

def d_crop(p):
    if not p: return None
    fi=np.asarray(p[0]["input"]); fo=np.asarray(p[0]["output"]); oh_,ow_=fo.shape; ih,iw=fi.shape
    if oh_>ih or ow_>iw: return None
    for r0 in range(ih-oh_+1):
        for c0 in range(iw-ow_+1):
            if not np.array_equal(fi[r0:r0+oh_,c0:c0+ow_],fo): continue
            if all(np.asarray(x["output"]).shape==(oh_,ow_) and r0+oh_<=np.asarray(x["input"]).shape[0] and
                   c0+ow_<=np.asarray(x["input"]).shape[1] and
                   np.array_equal(np.asarray(x["input"])[r0:r0+oh_,c0:c0+ow_],np.asarray(x["output"])) for x in p[1:]):
                return (r0,c0,oh_,ow_)
    return None

def d_shift(p):
    if not all_same(p): return None
    ih,iw=np.asarray(p[0]["input"]).shape
    for dr in range(-min(ih,11)+1,min(ih,11)):
        for dc in range(-min(iw,11)+1,min(iw,11)):
            if dr==0 and dc==0: continue
            ok=True
            for x in p:
                ig=np.asarray(x["input"]); og=np.asarray(x["output"]); hh,ww=ig.shape
                s=np.zeros_like(ig)
                sr0=max(0,-dr);sr1=min(hh,hh-dr);sc0=max(0,-dc);sc1=min(ww,ww-dc)
                dr0=max(0,dr);dc0=max(0,dc);rh=sr1-sr0;rw=sc1-sc0
                if rh>0 and rw>0: s[dr0:dr0+rh,dc0:dc0+rw]=ig[sr0:sr1,sc0:sc1]
                if not np.array_equal(s,og): ok=False; break
            if ok: return (dr,dc)
    return None

def d_cf(p):
    if not all_same(p): return None
    cand=None
    for x in p:
        ig=np.asarray(x["input"]); og=np.asarray(x["output"]); diff=(ig!=og)
        if not np.any(diff): continue
        if not np.all(og[diff]==0): return None
        cand=(cand or set())|set(int(v) for v in ig[diff].flat)
    if not cand: return None
    for x in p:
        ig=np.asarray(x["input"]).copy()
        for c_ in cand: ig[ig==c_]=0
        if not np.array_equal(ig,np.asarray(x["output"])): return None
    return cand

def d_up(p):
    for f in [2,3,4,5]:
        if all(np.asarray(x["output"]).shape==(np.asarray(x["input"]).shape[0]*f,np.asarray(x["input"]).shape[1]*f) and
               np.asarray(x["input"]).shape[0]*f<=H and np.asarray(x["input"]).shape[1]*f<=W and
               np.array_equal(np.repeat(np.repeat(np.asarray(x["input"]),f,0),f,1),np.asarray(x["output"])) for x in p): return f
    return None

def d_dn(p):
    for f in [2,3,4,5]:
        if all(np.asarray(x["input"]).shape[0]==np.asarray(x["output"]).shape[0]*f and
               np.array_equal(np.asarray(x["input"])[::f,::f],np.asarray(x["output"])) for x in p): return f
    return None

def d_tile(p):
    if not p: return None
    ih,iw=np.asarray(p[0]["input"]).shape; oh_,ow_=np.asarray(p[0]["output"]).shape
    if ih==0 or iw==0 or oh_%ih or ow_%iw: return None
    tr,tc=oh_//ih,ow_//iw
    if tr*ih>H or tc*iw>W or (tr==1 and tc==1): return None
    for x in p:
        ig=np.asarray(x["input"]); og=np.asarray(x["output"])
        if og.shape!=(ig.shape[0]*tr,ig.shape[1]*tc) or not np.array_equal(np.tile(ig,(tr,tc)),og): return None
    return (tr,tc)

def d_catfw(p):
    return bool(p) and all(np.asarray(x["output"]).shape==(np.asarray(x["input"]).shape[0],np.asarray(x["input"]).shape[1]*2) and
               np.array_equal(np.concatenate([np.asarray(x["input"]),np.fliplr(np.asarray(x["input"]))],1),np.asarray(x["output"])) for x in p)

def d_catfh(p):
    return bool(p) and all(np.asarray(x["output"]).shape==(np.asarray(x["input"]).shape[0]*2,np.asarray(x["input"]).shape[1]) and
               np.array_equal(np.concatenate([np.asarray(x["input"]),np.flipud(np.asarray(x["input"]))],0),np.asarray(x["output"])) for x in p)

def d_quad(p):
    if not p: return False
    for x in p:
        ig=np.asarray(x["input"]); og=np.asarray(x["output"]); ih,iw=ig.shape
        if og.shape!=(ih*2,iw*2): return False
        top=np.concatenate([ig,np.fliplr(ig)],1); bot=np.concatenate([np.flipud(ig),np.rot90(ig,2)],1)
        if not np.array_equal(np.concatenate([top,bot],0),og): return False
    return True

def d_rowbc(p):
    if not all_same(p): return None
    ch=None
    for x in p:
        ig=np.asarray(x["input"]); og=np.asarray(x["output"]); mt=None
        for r in range(ig.shape[0]):
            if np.all(og==ig[r,:]): mt=r; break
        if mt is None: return None
        if ch is None: ch=mt
        elif ch!=mt: return None
    return ch

def d_colbc(p):
    if not all_same(p): return None
    ch=None
    for x in p:
        ig=np.asarray(x["input"]); og=np.asarray(x["output"]); mt=None
        for c in range(ig.shape[1]):
            if np.all(og==ig[:,c:c+1]): mt=c; break
        if mt is None: return None
        if ch is None: ch=mt
        elif ch!=mt: return None
    return ch

def d_cropcolor(p):
    if not p: return None
    fi=np.asarray(p[0]["input"]); fo=np.asarray(p[0]["output"]); oh_,ow_=fo.shape; ih,iw=fi.shape
    if oh_>ih or ow_>iw: return None
    for r0 in range(ih-oh_+1):
        for c0 in range(iw-ow_+1):
            cm={}; ok=True
            for x in p:
                ig=np.asarray(x["input"]); og=np.asarray(x["output"])
                if og.shape!=(oh_,ow_) or r0+oh_>ig.shape[0] or c0+ow_>ig.shape[1]: ok=False; break
                reg=ig[r0:r0+oh_,c0:c0+ow_]
                for a,b in zip(reg.flat,og.flat):
                    a,b=int(a),int(b)
                    if a in cm and cm[a]!=b: ok=False; break
                    cm[a]=b
                if not ok: break
            if not ok: continue
            if all(k==v for k,v in cm.items()): continue
            vf=True
            for x in p:
                reg=np.asarray(x["input"])[r0:r0+oh_,c0:c0+ow_]; mp=reg.copy()
                for k,v in cm.items(): mp[reg==k]=v
                if not np.array_equal(mp,np.asarray(x["output"])): vf=False; break
            if vf: return (r0,c0,oh_,ow_,cm)
    return None

def d_border(p):
    if not p: return None
    fi=np.asarray(p[0]["input"]); fo=np.asarray(p[0]["output"]); ih,iw=fi.shape; oh_,ow_=fo.shape
    if (oh_-ih)!=(ow_-iw) or (oh_-ih)<=0 or (oh_-ih)%2: return None
    b=(oh_-ih)//2
    if not np.array_equal(fo[b:b+ih,b:b+iw],fi): return None
    vals=set()
    for s in [fo[:b,:],fo[-b:,:],fo[:,:b],fo[:,-b:]]: vals|=set(int(v) for v in s.flat)
    if len(vals)!=1: return None
    fill=int(vals.pop())
    for x in p[1:]:
        ig=np.asarray(x["input"]); og=np.asarray(x["output"]); ih2,iw2=ig.shape
        if og.shape!=(ih2+2*b,iw2+2*b) or not np.array_equal(og[b:b+ih2,b:b+iw2],ig): return None
        m=np.zeros(og.shape,dtype=bool); m[:b,:]=True; m[-b:,:]=True; m[:,:b]=True; m[:,-b:]=True
        if not np.all(og[m]==fill): return None
    return (b,fill)

def d_sym(p):
    if not all_same(p): return None
    def fold(a,b):
        o=np.zeros_like(a); an=a!=0; bn=b!=0
        if np.any(an&bn&(a!=b)): return None
        o[an]=a[an]; o[bn&~an]=b[bn&~an]; return o
    for ax in ["h","w","hw","diag"]:
        ok=True
        for x in p:
            ig=np.asarray(x["input"]); og=np.asarray(x["output"])
            if ax=="h": f=fold(ig,np.flipud(ig))
            elif ax=="w": f=fold(ig,np.fliplr(ig))
            elif ax=="hw":
                f=fold(ig,np.flipud(ig))
                if f is not None: f=fold(f,np.fliplr(ig))
                if f is not None: f=fold(f,np.flipud(np.fliplr(ig)))
            elif ax=="diag":
                if ig.shape[0]!=ig.shape[1]: ok=False; break
                f=fold(ig,ig.T)
            if f is None or not np.array_equal(f,og): ok=False; break
        if ok: return ax
    return None

def d_perm(p):
    if not all(np.asarray(x["input"]).shape==(H,W) and np.asarray(x["output"]).shape==(H,W) for x in p): return None
    ig=np.asarray(p[0]["input"]).flatten(); og=np.asarray(p[0]["output"]).flatten()
    gi=np.zeros(H*W,dtype=np.int64)
    for d in range(H*W):
        w=np.where(ig==og[d])[0]
        if len(w)!=1: return None
        gi[d]=w[0]
    for x in p[1:]:
        if not np.array_equal(np.asarray(x["input"]).flatten()[gi],np.asarray(x["output"]).flatten()): return None
    return gi

# ═══════════════════════════════════════════════════════════
# Orchestrator
# ═══════════════════════════════════════════════════════════

def _cands(train,all_p):
    fs=fsize(all_p); full=(fs==(30,30)); same=all_same(train)

    if d_id(train): yield "id",mk_identity

    rot=d_rot(train)
    if rot:
        bmap={"180":(mk_rot180,mkf_rot180),"90cw":(mk_rot90cw,mkf_r90cw),"90ccw":(mk_rot90ccw,mkf_r90ccw)}
        bf,bff=bmap[rot]
        if full: yield rot,bf
        if fs: h,w=fs; yield rot+"_f",(lambda h=h,w=w,bff=bff:bff(h,w))

    flip=d_flip(train)
    if flip:
        bmap={"h":(mk_flip_h,mkf_fh),"w":(mk_flip_w,mkf_fw),"both":(mk_flip_both,mkf_rot180)}
        bf,bff=bmap[flip]
        if full: yield "fl_"+flip,bf
        if fs: h,w=fs; yield "fl_"+flip+"_f",(lambda h=h,w=w,bff=bff:bff(h,w))

    if d_tr(train):
        if full: yield "tr",mk_transpose
        if fs: h,w=fs; yield "tr_f",(lambda h=h,w=w:mkf_tr(h,w))

    if d_atr(train) and full: yield "atr",mk_anti_tr

    cm=d_cm(train)
    if cm and same: yield "crm",(lambda cm=cm:mk_color_remap(cm))

    for rl,k in [("180",2),("90cw",-1),("90ccw",1)]:
        xm=d_xc(train,lambda ig,k=k:np.rot90(ig,k))
        if xm and same:
            bmap={"180":(mk_rot180,mkf_rot180),"90cw":(mk_rot90cw,mkf_r90cw),"90ccw":(mk_rot90ccw,mkf_r90ccw)}
            bf,bff=bmap[rl]
            if full: yield rl+"_c",(lambda bf=bf,xm=xm:mk_xform_color(bf,xm))
            if fs: h,w=fs; yield rl+"_fc",(lambda bff=bff,h=h,w=w,xm=xm:mk_xform_color(lambda bff=bff,h=h,w=w:bff(h,w),xm))

    for fl,fn in [("h",np.flipud),("w",np.fliplr),("both",lambda a:np.flipud(np.fliplr(a)))]:
        xm=d_xc(train,fn)
        if xm and same:
            bmap={"h":(mk_flip_h,mkf_fh),"w":(mk_flip_w,mkf_fw),"both":(mk_flip_both,mkf_rot180)}
            bf,bff=bmap[fl]
            if full: yield "fl_"+fl+"_c",(lambda bf=bf,xm=xm:mk_xform_color(bf,xm))
            if fs: h,w=fs; yield "fl_"+fl+"_fc",(lambda bff=bff,h=h,w=w,xm=xm:mk_xform_color(lambda bff=bff,h=h,w=w:bff(h,w),xm))

    tc=d_xc(train,lambda ig:ig.T)
    if tc:
        if full: yield "tr_c",(lambda tc=tc:mk_xform_color(mk_transpose,tc))
        if fs: h,w=fs; yield "tr_fc",(lambda h=h,w=w,tc=tc:mk_xform_color(lambda h=h,w=w:mkf_tr(h,w),tc))

    cf=d_cf(train)
    if cf and same: yield "cf",(lambda cm={c_:0 for c_ in cf}:mk_color_remap(cm))

    crop=d_crop(train)
    if crop: r0,c0,oh_,ow_=crop; yield "crop",(lambda r0=r0,c0=c0,oh_=oh_,ow_=ow_:mk_crop_slice(r0,c0,oh_,ow_))

    sh=d_shift(train)
    if sh:
        dr,dc=sh
        if full: yield "sh",(lambda dr=dr,dc=dc:mk_ftranslate(H,W,dr,dc))
        if fs: h,w=fs; yield "sh_f",(lambda h=h,w=w,dr=dr,dc=dc:mk_ftranslate(h,w,dr,dc))

    if fs:
        h,w=fs
        up=d_up(train)
        if up: yield "up",(lambda f=up,h=h,w=w:mk_upscale(f,h,w))
        dn=d_dn(train)
        if dn: yield "dn",(lambda f=dn,h=h,w=w:mk_downscale(f,h,w))
        tl=d_tile(train)
        if tl: tr_,tc_=tl; yield "tile",(lambda tr_=tr_,tc_=tc_,h=h,w=w:mk_tile(tr_,tc_,h,w))
        if d_catfw(train) and w*2<=W: yield "catfw",(lambda h=h,w=w:mk_cat_fw(h,w))
        if d_catfh(train) and h*2<=H: yield "catfh",(lambda h=h,w=w:mk_cat_fh(h,w))
        if d_quad(train) and h*2<=H and w*2<=W: yield "quad",(lambda h=h,w=w:mk_quad(h,w))
        rb=d_rowbc(train)
        if rb is not None: yield "rb",(lambda h=h,w=w,rb=rb:mk_row_bc(h,w,rb))
        cb=d_colbc(train)
        if cb is not None: yield "cb",(lambda h=h,w=w,cb=cb:mk_col_bc(h,w,cb))
        sym=d_sym(train)
        if sym:
            sb={"h":mk_sym_h,"w":mk_sym_w,"hw":mk_sym_hw,"diag":mk_sym_diag}
            if not(sym=="diag" and h!=w): yield "sym_"+sym,(lambda h=h,w=w,sym=sym:sb[sym](h,w))
        ba=d_border(train)
        if ba:
            b,fill=ba
            if h+2*b<=H and w+2*b<=W: yield "brd",(lambda h=h,w=w,b=b,fill=fill:mk_border(h,w,b,fill))

    ccp=d_cropcolor(train)
    if ccp: r0,c0,oh_,ow_,cmap=ccp; yield "crpcol",(lambda r0=r0,c0=c0,oh_=oh_,ow_=ow_,cmap=cmap:mk_crop_color(r0,c0,oh_,ow_,cmap))

    perm=d_perm(train)
    if perm is not None:
        gi=perm
        def _mk_perm(gi=gi):
            g=oh.make_graph(
                [oh.make_node("Reshape",["input","sc"],["fl"]),oh.make_node("Gather",["fl","gi"],["gd"],axis=2),
                 oh.make_node("Reshape",["gd","so"],["output"])],
                "perm",[_X()],[_Y()],
                [onh.from_array(np.array([1,C,H*W],dtype=np.int64),name="sc"),
                 onh.from_array(gi.astype(np.int64),name="gi"),
                 onh.from_array(np.array([1,C,H,W],dtype=np.int64),name="so")])
            mm=oh.make_model(g,opset_imports=OPSET); mm.ir_version=8; return mm
        yield "perm",_mk_perm

    cst=d_const(train)
    if cst is not None: yield "const",(lambda k=cst:mk_const(k))


def solve_task(task_data,task_name=""):
    train=task_data.get("train",[]) or []; test=task_data.get("test",[]) or []
    ag=task_data.get("arc-gen",[]) or []; ags=ag[:10]
    all_p=train+test+ags
    if not all_p or not train: return None,None,None

    best=None; bc=float("inf"); bl=None

    for label,builder in _cands(train,all_p):
        try: m=builder()
        except: continue
        if m is None or not check_fits(m) or not check_ops(m): continue
        if not check_correct(m,all_p): continue
        c=cost(m)
        if c<bc: bc=c; best=m; bl=label

    sp=solve_spatial(task_data)
    if sp:
        c=cost(sp)
        if c<bc: bc=c; best=sp; bl="spatial"

    if best is None and all_same(train):
        for ks in [1,3,5]:
            m=try_conv(train,ks,800)
            if m and check_correct(m,all_p):
                c=cost(m)
                if c<bc: bc=c; best=m; bl=f"conv{ks}"; break
        if best is None:
            m=try_2layer(train)
            if m and check_correct(m,all_p):
                c=cost(m)
                if c<bc: bc=c; best=m; bl="conv2l"

    return best,bl,bc

def load_task(path):
    with open(path) as f: return json.load(f)

def save_model(model, path):
    try:
        onnx.checker.check_model(model)
    except Exception:
        pass
    with open(path, "wb") as f:
        f.write(model.SerializeToString())


def find_task_files():
    candidates = [
        Path("/kaggle/input/competitions/neurogolf-2026"),
        Path("."),
    ]
    for d in candidates:
        if d.exists():
            files = sorted(d.glob("task*.json"))
            if files:
                return files
    task_files = []
    root_base = "/kaggle/input" if Path("/kaggle/input").exists() else "."
    for root, _, files in os.walk(root_base):
        for fname in files:
            if fname.startswith("task") and fname.endswith(".json"):
                task_files.append(Path(root) / fname)
    return sorted(task_files)

In [ ]:
import starter_solver as starter
import fast_solver as fast

def load_neurogolf_utils(task_dir: Path):
    utils_path = task_dir / 'neurogolf_utils' / 'neurogolf_utils.py'
    spec = importlib.util.spec_from_file_location('neurogolf_utils_local', utils_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Unable to load helper module from {utils_path}')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


UTILS = load_neurogolf_utils(TASK_DIR)


def full_public_check(namespace_module, model: onnx.ModelProto, task_data: dict) -> bool:
    all_pairs = task_data.get('train', []) + task_data.get('test', []) + task_data.get('arc-gen', [])
    try:
        session = ort.InferenceSession(model.SerializeToString(), providers=['CPUExecutionProvider'])
    except Exception:
        return False
    for pair in all_pairs:
        input_tensor = namespace_module.grid_to_tensor(pair['input'])
        output_grid = pair['output']
        output_h = len(output_grid)
        output_w = len(output_grid[0]) if output_grid else 0
        try:
            prediction = session.run(['output'], {'input': input_tensor})[0]
        except Exception:
            return False
        predicted_grid = namespace_module.tensor_to_grid(prediction, output_h, output_w)
        if predicted_grid != output_grid:
            return False
    return True


def validate_and_score_payload(filename: str, payload: bytes) -> dict:
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / filename
        path.write_bytes(payload)
        model = onnx.load(str(path))
        onnx.checker.check_model(model)
        session = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
        sample = np.zeros((1, 10, 30, 30), dtype=np.float32)
        _ = session.run(['output'], {'input': sample})[0]
        macs, memory, params = UTILS.score_network(str(path))
        total_cost = int(macs + memory + params)
        return {
            'filesize_bytes': int(path.stat().st_size),
            'macs': int(macs),
            'memory': int(memory),
            'params': int(params),
            'total_cost': total_cost,
        }


def make_small_mask_gather_model(gather_indices: np.ndarray, mask_hw: np.ndarray, name: str) -> onnx.ModelProto:
    shape_chw = onh.from_array(np.array([1, 10, 900], dtype=np.int64), name='shape_chw')
    shape_out = onh.from_array(np.array([1, 10, 30, 30], dtype=np.int64), name='shape_out')
    gather_tensor = onh.from_array(gather_indices.astype(np.int64), name='gather_idx')
    mask_tensor = onh.from_array(mask_hw.astype(np.float32), name='mask')
    nodes = [
        oh.make_node('Reshape', ['input', 'shape_chw'], ['flat']),
        oh.make_node('Gather', ['flat', 'gather_idx'], ['flat_gathered'], axis=2),
        oh.make_node('Reshape', ['flat_gathered', 'shape_out'], ['gathered']),
        oh.make_node('Mul', ['gathered', 'mask'], ['output']),
    ]
    graph = oh.make_graph(
        nodes,
        name,
        [oh.make_tensor_value_info('input', onnx.TensorProto.FLOAT, [1, 10, 30, 30])],
        [oh.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [1, 10, 30, 30])],
        [shape_chw, shape_out, gather_tensor, mask_tensor],
    )
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    model.ir_version = 8
    return model


def infer_zero_ambiguity_spatial(task_data: dict) -> tuple[onnx.ModelProto | None, int | None]:
    pairs = task_data.get('train', []) + task_data.get('test', []) + task_data.get('arc-gen', [])
    for pair in pairs:
        if (
            len(pair['input']) > 30
            or len(pair['input'][0]) > 30
            or len(pair['output']) > 30
            or len(pair['output'][0]) > 30
        ):
            return None, None

    inputs = []
    outputs = []
    for pair in pairs:
        inputs.append(starter.grid_to_tensor(pair['input'])[0].reshape(10, -1))
        outputs.append(starter.grid_to_tensor(pair['output'])[0].reshape(10, -1))
    inputs = np.stack(inputs, axis=0)
    outputs = np.stack(outputs, axis=0)

    input_lookup: dict[bytes, list[int]] = {}
    for source_index in range(900):
        signature = inputs[:, :, source_index].tobytes()
        input_lookup.setdefault(signature, []).append(source_index)

    gather_indices = np.zeros((900,), dtype=np.int64)
    mask_hw = np.zeros((1, 1, 30, 30), dtype=np.float32)
    ambiguous_positions = 0

    for output_index in range(900):
        signature_array = outputs[:, :, output_index]
        if np.all(signature_array == 0.0):
            continue
        candidates = input_lookup.get(signature_array.tobytes())
        if not candidates:
            return None, None
        if len(candidates) > 1:
            ambiguous_positions += 1
        source_index = candidates[0]
        gather_indices[output_index] = source_index
        row, col = divmod(output_index, 30)
        mask_hw[0, 0, row, col] = 1.0

    model = make_small_mask_gather_model(gather_indices, mask_hw, 'spatial_copy_zero_ambiguity')
    if not full_public_check(starter, model, task_data):
        return None, None
    return model, ambiguous_positions


def export_candidate(task_id: str, label: str, source: str, model: onnx.ModelProto, save_fn, extra: dict | None = None) -> dict:
    with tempfile.TemporaryDirectory() as tmpdir:
        path = Path(tmpdir) / f'{task_id}_{source}.onnx'
        save_fn(model, path)
        payload = path.read_bytes()
    details = validate_and_score_payload(f'{task_id}_{source}.onnx', payload)
    record = {
        'task_id': task_id,
        'label': label,
        'source': source,
        'payload': payload,
        **details,
    }
    if extra:
        record.update(extra)
    return record


def choose_best(candidates: list[dict]) -> dict | None:
    if not candidates:
        return None
    return min(candidates, key=lambda item: item['total_cost'])


task_paths = sorted(TASK_DIR.glob('task*.json'))
print(f'Found {len(task_paths)} tasks')

rows = []
solver_counts = Counter()
output_paths = []

for index, task_path in enumerate(task_paths, start=1):
    task_id = task_path.stem
    task_data = starter.load_task(task_path)
    candidates = []

    try:
        baseline_model = starter.solve_task(task_data, task_id)
        if baseline_model is not None and full_public_check(starter, baseline_model, task_data):
            candidates.append(export_candidate(task_id, 'starter', 'starter', baseline_model, starter.save_model))
    except Exception as exc:
        pass

    try:
        fast_output = fast.solve_task(task_data, task_id)
        if isinstance(fast_output, tuple):
            fast_model = fast_output[0]
            fast_label = str(fast_output[1]) if len(fast_output) > 1 else 'fast_public'
        else:
            fast_model = fast_output
            fast_label = 'fast_public'
        if fast_model is not None and full_public_check(fast, fast_model, task_data):
            candidates.append(export_candidate(task_id, fast_label, 'fast_public', fast_model, fast.save_model))
    except Exception:
        pass

    spatial_model, ambiguous_positions = infer_zero_ambiguity_spatial(task_data)
    if spatial_model is not None and ambiguous_positions == 0:
        candidates.append(
            export_candidate(
                task_id,
                'spatial_zero_ambiguity',
                'spatial_zero_ambiguity',
                spatial_model,
                starter.save_model,
                extra={'ambiguous_positions': ambiguous_positions},
            )
        )

    best = choose_best(candidates)
    if best is None:
        continue

    out_path = WORKDIR / f'{task_id}.onnx'
    out_path.write_bytes(best.pop('payload'))
    best['model_path'] = str(out_path)
    rows.append(best)
    output_paths.append(out_path)
    solver_counts[best['source']] += 1

    if index % 25 == 0 or index == len(task_paths):
        print(f'[{index:03d}/{len(task_paths):03d}] solved so far: {len(rows)}')

rows.sort(key=lambda row: row['task_id'])
submission_csv = WORKDIR / 'submission.csv'
fieldnames = []
for row in rows:
    for key in row.keys():
        if key not in fieldnames:
            fieldnames.append(key)
with submission_csv.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

submission_zip = WORKDIR / 'submission.zip'
with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in output_paths:
        zf.write(path, path.name)

proxy_score = sum(max(0.0, 25.0 - math.log(int(row['total_cost']))) for row in rows)
summary = {
    'task_count': len(rows),
    'proxy_score': proxy_score,
    'solver_counts': dict(solver_counts),
    'submission_csv': str(submission_csv),
    'submission_zip': str(submission_zip),
}
summary_path = WORKDIR / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))

In [ ]:
from IPython.display import FileLink, display
for rel in ['submission.zip', 'submission.csv', 'summary.json']:
    path = WORKDIR / rel
    if path.exists():
        display(FileLink(path))